In [ ]:
import os
import sys

# Make the project's src package available from the notebooks folder.
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import data, wearables, baselines, evaluation

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
SEED = 0

Split train and test the same as baselines.

In [ ]:
# Modelling setup: same target, cycle rules and participant split as 02_baselines.
TARGET = "days_until_next_period"
# TODO(INTERVAL2_REVIEW): Expand only after the authors' response is reviewed.
# Recheck HR aggregation in src/wearables.py and Interval 2 phase-derived labels.
MODEL_INTERVALS = (2022,)

baseline_diary = data.load_diary()
baseline_starts = data.period_starts(baseline_diary, gap_days=3)
baseline_cycles = data.build_cycles(baseline_starts, min_len=15, max_len=60)

# Split BEFORE restricting intervals: preserve baseline participant assignments.
model_split = data.participant_split(baseline_cycles, test_frac=0.3, seed=SEED)
model_cycles = baseline_cycles.loc[
    baseline_cycles.study_interval.isin(MODEL_INTERVALS)
].copy()
model_frame = data.make_daily_prediction_frame(model_cycles)
model_train_cycles, model_test_cycles = model_split.apply(model_cycles)
model_train_frame, model_test_frame = model_split.apply(model_frame)

assert set(model_train_frame.id).isdisjoint(model_test_frame.id)
assert model_frame.study_interval.isin(MODEL_INTERVALS).all()
assert TARGET in model_frame.columns

# TODO(TIMING_ALIGNMENT): Before joining rolling predictors, settle the cutoff.
# Temperature uses sleep-end day; breathing rate is timestamped at wake time.
# HRV uses night-start labels (before noon -> previous day); daily HR availability is undocumented.
# The exploratory plots include today's readings: do not reuse them blindly.
# Refit/evaluate baselines on these same Interval 1 rows for a fair comparison.
# Keep feature selection inside the training participants; reserve test participants.
print(f"Target: {TARGET}; intervals: {MODEL_INTERVALS}")
print(f"Train: {model_train_frame.id.nunique()} participants, {len(model_train_frame):,} rows")
print(f"Test: {model_test_frame.id.nunique()} participants, {len(model_test_frame):,} rows")


## Baselines

Do ridge regression, random forest with cycle day and prior-cycle history to create baselines to understand the value added by features

In [ ]:
# History available at prediction time; no current-cycle outcome is a predictor.
# Only earlier cycles passing the existing cycle-length filters are included.
def add_prior_cycle_history(frame, cycles):
    out = frame.drop(
        columns=["has_prior_cycle", "previous_cycle_length"], errors="ignore",
    ).copy()
    out["has_prior_cycle"] = False
    out["previous_cycle_length"] = np.nan
    for (participant, interval), rows in out.groupby(["id", "study_interval"]):
        history = cycles.loc[
            cycles.id.eq(participant) & cycles.study_interval.eq(interval)
        ].sort_values("next_start_day")
        if history.empty:
            continue
        completed_days = history.next_start_day.to_numpy()
        # A cycle ending on D is complete once the new period is logged on D,
        # matching the existing assumption that today's cycle_day is known.
        positions = np.searchsorted(completed_days, rows.day_in_study.to_numpy(), side="right") - 1
        eligible = positions >= 0
        indices = rows.index[eligible]
        prior = history.iloc[positions[eligible]]
        assert (prior.next_start_day.to_numpy() <= rows.loc[indices, "day_in_study"].to_numpy()).all()
        out.loc[indices, "has_prior_cycle"] = True
        out.loc[indices, "previous_cycle_length"] = prior.cycle_length.to_numpy()
    return out

history_frame = add_prior_cycle_history(model_frame, model_cycles)
history_train_frame, history_test_frame = model_split.apply(history_frame)
history_feature_columns = ["cycle_day", "has_prior_cycle", "previous_cycle_length"]
assert len(history_frame) == len(model_frame)
assert history_frame.loc[~history_frame.has_prior_cycle, "previous_cycle_length"].isna().all()
assert set(history_train_frame.id).isdisjoint(history_test_frame.id)

# Keep a history-only frame for comparison with history plus wearable features.
# Missing prior length is NOT zero; impute inside each model's training folds.
history_train_frame.groupby("has_prior_cycle").agg(
    prediction_rows=("id", "size"),
    participants=("id", "nunique"),
)


In [ ]:
# Ridge using cycle day + prior-cycle history ONLY; no wearable features.
# Interval 1, same 20 participant splits as the other model comparisons.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV
from IPython.display import display

def evaluate_history_only_ridge():
    columns = ["cycle_day", "has_prior_cycle", "previous_cycle_length"]
    pipeline = Pipeline([
        # The flag already identifies missing history; no extra indicator needed.
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
        ("ridge", Ridge()),
    ])
    scores, parameters = [], []
    for seed in range(20):
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(history_frame)
        assert train.study_interval.eq(2022).all() and test.study_interval.eq(2022).all()
        assert set(train.id).isdisjoint(test.id)
        if train.id.nunique() < 2 or test.empty:
            raise ValueError("Insufficient participants for nested evaluation.")
        search = GridSearchCV(
            pipeline,
            param_grid={"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
            scoring="neg_mean_absolute_error",
            cv=GroupKFold(n_splits=min(5, train.id.nunique())),
            refit=True, error_score="raise",
        )
        # Imputation/scaling/alpha selection use inner training participants only.
        search.fit(train[columns], train[TARGET], groups=train.id)
        predictions = search.predict(test[columns])  # Raw predictions, like wearable ridge.
        assert np.isfinite(predictions).all()
        error = np.abs(test[TARGET].to_numpy() - predictions)
        scores.append({
            "seed": seed, "model": "ridge-history-only",
            "MAE": error.mean(),
            "participant_mean_MAE": pd.Series(
                error, index=test.id.to_numpy()
            ).groupby(level=0).mean().mean(),
        })
        parameters.append({"seed": seed, "alpha": search.best_params_["ridge__alpha"]})
        if (seed + 1) % 5 == 0:
            print(f"History-only ridge: completed {seed + 1}/20 splits")
    return pd.DataFrame(scores), pd.DataFrame(parameters)

history_ridge_scores, history_ridge_alphas = evaluate_history_only_ridge()
history_ridge_summary = history_ridge_scores.groupby("model").agg(
    mean_MAE_days=("MAE", "mean"),
    split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
)
display(history_ridge_summary.round(3))
display(history_ridge_alphas.alpha.value_counts().sort_index().rename("n_splits").to_frame())
# Split SD is variability across overlapping splits, not a confidence interval.
# Prior history for held-out participants is allowed only once completed.
# No current-cycle length, wearable values, or participant IDs enter the predictors.
# Exploratory comparison; no files exported. Clear outputs before committing.


In [ ]:
# History-only random forest: cycle day, has_prior_cycle, previous_cycle_length.
# Interval 1; same 20 outer splits and tree search as wearable random forest.
# Exploratory repeated evaluation; not an untouched final holdout.
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, GroupKFold
from IPython.display import display

def evaluate_history_only_forest():
    # Scaling is unnecessary for trees. Imputation is fitted inside each fold.
    pipeline = Pipeline([
        ("impute", SimpleImputer(
            strategy="median", keep_empty_features=True,
        )),
        ("forest", RandomForestRegressor(
            n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=-1,
        )),
    ])
    # Small, predefined search: limit depth and require multiple rows per leaf.
    # Leaves contain rows, not necessarily distinct participants.
    parameter_grid = {
        "forest__max_depth": [3, 6],
        "forest__min_samples_leaf": [10, 30],
    }
    score_rows, parameter_rows = [], []
    for seed in range(20):
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(history_frame)
        assert train.study_interval.eq(2022).all() and test.study_interval.eq(2022).all()
        assert set(train.id).isdisjoint(test.id)
        if train.id.nunique() < 2 or test.empty:
            raise ValueError("Insufficient participants for nested evaluation.")
        search = GridSearchCV(
            pipeline, param_grid=parameter_grid,
            scoring="neg_mean_absolute_error",
            cv=GroupKFold(n_splits=min(5, train.id.nunique())),
            refit=True, error_score="raise", n_jobs=1,
        )
        search.fit(train[history_feature_columns], train[TARGET], groups=train.id)
        predicted = search.predict(test[history_feature_columns])
        assert np.isfinite(predicted).all()
        error = np.abs(test[TARGET].to_numpy() - predicted)
        score_rows.append({
            "seed": seed, "model": "random-forest-history-only",
            "MAE": error.mean(),
            "participant_mean_MAE": pd.Series(
                error, index=test.id.to_numpy()
            ).groupby(level=0).mean().mean(),
        })
        parameter_rows.append({
            "seed": seed,
            "max_depth": search.best_params_["forest__max_depth"],
            "min_samples_leaf": search.best_params_["forest__min_samples_leaf"],
        })
        if (seed + 1) % 5 == 0:
            print(f"Random forest: completed {seed + 1}/20 splits")
    return pd.DataFrame(score_rows), pd.DataFrame(parameter_rows)

history_rf_scores, history_rf_parameters = evaluate_history_only_forest()
history_rf_summary = history_rf_scores.groupby("model").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
)
display(history_rf_summary.round(3))
display(history_rf_parameters.groupby(["max_depth", "min_samples_leaf"]).size().rename("n_splits").to_frame())


## Modelling

Start with ridge regression (preferred to linear regression for dependent variable overfitting). Then random forest. Suggested from [Kilungeja et al.](https://www.nature.com/articles/s44294-025-00078-8). 6-day windows first, then 3 day windows.

In [ ]:
# Six completed calendar days: D-6 through D-1, never prediction day D.
WINDOW_DAYS = 6
MIN_VALID_DAYS = 6  # Match exploration: incomplete windows remain NaN.
six_day_feature_columns = [
    f"{signal}_6d_{stat}"
    for signal in wearables.SIGNALS
    for stat in ("median", "sd", "slope", "min", "max")
]

def build_six_day_features(daily, prediction_keys):
    # Include prediction dates so windows also work after the final recorded day.
    keys = pd.concat([daily[wearables.KEY], prediction_keys[wearables.KEY]])
    keys = keys.drop_duplicates()
    grid = wearables.dense_days(keys)
    grid = grid.merge(daily, on=wearables.KEY, how="left", validate="one_to_one")
    pieces = []
    for _, person in grid.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        result = person[wearables.KEY].copy()
        for signal in wearables.SIGNALS:
            # HRV night N includes morning N+1. Extra lag avoids prediction-day readings.
            # HRV uses night labels D-7..D-2; wake-day signals use D-6..D-1.
            previous = person[signal].shift(2 if signal == "hrv_rmssd" else 1)
            window = previous.rolling(WINDOW_DAYS, min_periods=MIN_VALID_DAYS)
            prefix = f"{signal}_6d_"
            result[prefix + "median"] = window.median()
            result[prefix + "sd"] = window.std(ddof=1)
            # Dense calendar grid: x positions correspond to days, including gaps.
            result[prefix + "slope"] = window.apply(
                lambda values: np.polyfit(np.arange(len(values)), values, 1)[0],
                raw=True,
            )
            result[prefix + "min"] = window.min()
            result[prefix + "max"] = window.max()
        pieces.append(result)
    return pd.concat(pieces, ignore_index=True)

# Existing loader: daily HR minimum, temperature/HRV/breathing-rate means.
# This differs from the daily medians used in exploratory plots.
# TODO(INTERVAL2_REVIEW): Keep MODEL_INTERVALS restricted pending author feedback.
# TODO(TIMING_ALIGNMENT): Verify the noon-based HRV night-start rule against sleep sessions.
# TODO(CLEANING_REVIEW): Validate the loader's RMSSD upper threshold before fitting.
daily_model_signals = wearables.load_daily_signals()
daily_model_signals = daily_model_signals.loc[
    daily_model_signals.study_interval.isin(MODEL_INTERVALS)
].copy()
six_day_features = build_six_day_features(daily_model_signals, model_frame)

# Left join preserves all prediction rows; no imputation or scaling here.
# Any learned preprocessing must later be fitted within training folds only.
model_frame_6d = history_frame.merge(
    six_day_features[wearables.KEY + six_day_feature_columns],
    on=wearables.KEY, how="left", validate="one_to_one",
)
model_train_6d, model_test_6d = model_split.apply(model_frame_6d)
assert len(model_frame_6d) == len(model_frame)
assert len(six_day_feature_columns) == 20
assert set(model_train_6d.id).isdisjoint(model_test_6d.id)

# Inspect coverage on TRAINING rows only; do not select features using test results.
six_day_coverage = pd.DataFrame({
    "n_available": model_train_6d[six_day_feature_columns].notna().sum(),
    "pct_missing": 100 * model_train_6d[six_day_feature_columns].isna().mean(),
})
six_day_coverage.round(2)


In [ ]:
# Ridge: cycle day + prior-cycle history + six-day wearable features (no symptoms).
# Exploratory model: loader alignment/cleaning TODOs above still need validation.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold, GridSearchCV

ridge_feature_columns = history_feature_columns + six_day_feature_columns
assert len(ridge_feature_columns) == 23
# Explicit allowlist excludes cycle_length, which would reveal the answer.
X_ridge_train = model_train_6d[ridge_feature_columns]
y_ridge_train = model_train_6d[TARGET]
ridge_groups = model_train_6d["id"]

ridge_pipeline = Pipeline([
    # Medians are learned only from each fold's training participants.
    # Missingness indicators let the model distinguish imputed readings.
    # Entirely missing columns are retained (filled with zero).
    ("impute", SimpleImputer(
        strategy="median", add_indicator=True, keep_empty_features=True,
    )),
    ("scale", StandardScaler()),
    ("ridge", Ridge()),
])

n_ridge_folds = min(5, ridge_groups.nunique())
if n_ridge_folds < 2:
    raise ValueError("Ridge validation requires at least two training participants.")
ridge_cv_splits = list(GroupKFold(n_splits=n_ridge_folds).split(
    X_ridge_train, y_ridge_train, groups=ridge_groups,
))
for train_idx, validation_idx in ridge_cv_splits:
    assert set(ridge_groups.iloc[train_idx]).isdisjoint(
        ridge_groups.iloc[validation_idx]
    )

ridge_search = GridSearchCV(
    ridge_pipeline,
    param_grid={"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
    scoring="neg_mean_absolute_error",
    cv=ridge_cv_splits,
    refit=True,
    error_score="raise",
)
ridge_search.fit(X_ridge_train, y_ridge_train)
ridge_model = ridge_search.best_estimator_  # Refit on all training participants.

# Alpha controls shrinkage: larger values shrink coefficients more.
# These are tuning scores, not an unbiased final test evaluation.
# MAE is computed per row within each fold, then averaged across folds.
# Raw regression predictions are used here (no clipping at zero).
ridge_cv_results = pd.DataFrame({
    "alpha": ridge_search.cv_results_["param_ridge__alpha"],
    "validation_MAE_days": -ridge_search.cv_results_["mean_test_score"],
    "fold_SD_days": ridge_search.cv_results_["std_test_score"],
}).sort_values("validation_MAE_days")
print(f"Selected alpha: {ridge_search.best_params_['ridge__alpha']}")
print("Test participants remain untouched; no test predictions generated.")
ridge_cv_results.round(3)


In [ ]:
# Repeated outer tests: same baseline seeds (0..19), restricted to Interval 1.
# This now reuses the original holdout: results are exploratory if used to
# choose later features/models, not an untouched final performance estimate.
from sklearn.base import clone
from IPython.display import display

repeated_rows = []
ridge_alpha_rows = []
for split_seed in range(20):
    # Preserve the original baseline participant assignments before filtering.
    outer_split = data.participant_split(baseline_cycles, test_frac=0.3, seed=split_seed)
    tr_cycles, te_cycles = outer_split.apply(model_cycles)
    tr_frame, te_frame = outer_split.apply(model_frame_6d)
    assert set(tr_frame.id).isdisjoint(te_frame.id)
    if tr_frame.id.nunique() < 2 or te_frame.empty:
        raise ValueError("Insufficient participants for nested evaluation.")

    # Tune alpha afresh; all preprocessing stays inside the inner training folds.
    inner_cv = GroupKFold(n_splits=min(5, tr_frame.id.nunique()))
    search = GridSearchCV(
        clone(ridge_pipeline),
        param_grid={"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
        scoring="neg_mean_absolute_error",
        cv=inner_cv, refit=True, error_score="raise",
    )
    search.fit(tr_frame[ridge_feature_columns], tr_frame[TARGET], groups=tr_frame.id)

    pop = baselines.PopulationBaseline(statistic="median").fit(tr_cycles)
    cal = baselines.CalendarDayModel().fit(tr_frame)
    # The existing personalised baseline uses ONLY earlier completed cycles
    # of each held-out participant; otherwise it falls back to training population.
    personal = baselines.PersonalBaseline(statistic="median", fallback=pop).fit(te_cycles)
    predictions = {
        "population": pop.predict(te_frame),
        "calendar-day": cal.predict(te_frame),
        "personalised": personal.predict(te_frame),
        # Keep raw ridge predictions, matching the preceding tuning cell.
        "ridge-6d": search.predict(te_frame[ridge_feature_columns]),
    }
    for name, pred in predictions.items():
        assert np.isfinite(pred).all()
        absolute_error = np.abs(te_frame[TARGET].to_numpy() - pred)
        repeated_rows.append({
            "seed": split_seed, "model": name,
            "MAE": absolute_error.mean(),
            "participant_mean_MAE": pd.Series(
                absolute_error, index=te_frame.id.to_numpy()
            ).groupby(level=0).mean().mean(),
        })
    ridge_alpha_rows.append({
        "seed": split_seed,
        "alpha": search.best_params_["ridge__alpha"],
    })
    if (split_seed + 1) % 5 == 0:
        print(f"Completed {split_seed + 1}/20 splits")

repeated_model_scores = pd.DataFrame(repeated_rows)
repeated_ridge_alphas = pd.DataFrame(ridge_alpha_rows)
repeated_model_summary = repeated_model_scores.groupby("model").agg(
    mean_MAE_days=("MAE", "mean"),
    split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
).sort_values("mean_MAE_days")
# Paired differences compare the same test rows in each split.
# Negative difference means ridge has lower MAE than that baseline.
split_mae = repeated_model_scores.pivot(index="seed", columns="model", values="MAE")
ridge_baseline_comparison = pd.DataFrame([
    {
        "baseline": name,
        "mean_ridge_minus_baseline_days": (split_mae["ridge-6d"] - split_mae[name]).mean(),
        "ridge_better_splits": int((split_mae["ridge-6d"] < split_mae[name]).sum()),
        "n_splits": len(split_mae),
    }
    for name in ("population", "calendar-day", "personalised")
]).set_index("baseline")
print("Split SD measures variability, NOT a confidence interval; splits overlap.")
display(repeated_model_summary.round(3))
display(ridge_baseline_comparison.round(3))
display(repeated_ridge_alphas.alpha.value_counts().sort_index().rename("n_splits").to_frame())
# No files exported; clear outputs before committing.


In [ ]:
# # Sensitivity analysis ONLY: Interval 2 HR and phase-derived labels remain under review.
# # Earlier cells and their results are unchanged. Same 20 outer participant splits.
# # No interval indicator is added; only the amount/source of training data changes.
# def compare_both_intervals():
#     both_cycles = baseline_cycles.loc[baseline_cycles.study_interval.isin([2022, 2024])].copy()
#     both_frame = add_prior_cycle_history(data.make_daily_prediction_frame(both_cycles), both_cycles)
#     both_daily = wearables.load_daily_signals()
#     both_daily = both_daily.loc[both_daily.study_interval.isin([2022, 2024])].copy()
#     both_features = build_six_day_features(both_daily, both_frame)
#     both_frame = both_frame.merge(
#         both_features[wearables.KEY + six_day_feature_columns],
#         on=wearables.KEY, how="left", validate="one_to_one",
#     )
#     # Confirm that adding Interval 2 has not changed Interval 1 feature values.
#     cols = wearables.KEY + ridge_feature_columns + [TARGET]
#     pd.testing.assert_frame_equal(
#         both_frame.loc[both_frame.study_interval.eq(2022), cols]
#         .sort_values(wearables.KEY).reset_index(drop=True),
#         model_frame_6d[cols].sort_values(wearables.KEY).reset_index(drop=True),
#     )
#     rows, alpha_rows = [], []
#     for seed in range(20):
#         split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
#         trc, tec = split.apply(both_cycles)
#         trf, tef = split.apply(both_frame)
#         assert set(trf.id).isdisjoint(tef.id)
#         search = GridSearchCV(
#             clone(ridge_pipeline),
#             param_grid={"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
#             scoring="neg_mean_absolute_error",
#             cv=GroupKFold(n_splits=min(5, trf.id.nunique())),
#             refit=True, error_score="raise",
#         )
#         search.fit(trf[ridge_feature_columns], trf[TARGET], groups=trf.id)
#         pop = baselines.PopulationBaseline(statistic="median").fit(trc)
#         cal = baselines.CalendarDayModel().fit(trf)
#         # cycle_index resets per interval. Fit histories separately to prevent
#         # later-interval cycles leaking into earlier-interval personal predictions.
#         # No personal history is carried across the multi-year interval gap.
#         personal_pred = pd.Series(index=tef.index, dtype=float)
#         for interval, part in tef.groupby("study_interval"):
#             personal = baselines.PersonalBaseline(statistic="median", fallback=pop).fit(
#                 tec.loc[tec.study_interval.eq(interval)]
#             )
#             personal_pred.loc[part.index] = personal.predict(part)
#         preds = {
#             "population": pop.predict(tef),
#             "calendar-day": cal.predict(tef),
#             "personalised": personal_pred.to_numpy(),
#             "ridge-6d": search.predict(tef[ridge_feature_columns]),
#         }
#         for name, pred in preds.items():
#             assert np.isfinite(pred).all()
#             error = pd.Series(np.abs(tef[TARGET].to_numpy() - pred), index=tef.index)
#             for scope, mask in {
#                 "Both intervals": np.ones(len(tef), dtype=bool),
#                 "Interval 1 only": tef.study_interval.eq(2022).to_numpy(),
#                 "Interval 2 only": tef.study_interval.eq(2024).to_numpy(),
#             }.items():
#                 if not mask.any():
#                     continue
#                 rows.append({
#                     "seed": seed, "model": name, "test_scope": scope,
#                     "MAE": error.loc[mask].mean(),
#                     "n_test_rows": int(mask.sum()),
#                     "participant_mean_MAE": error.loc[mask].groupby(tef.loc[mask, "id"]).mean().mean(),
#                 })
#         alpha_rows.append({"seed": seed, "alpha": search.best_params_["ridge__alpha"]})
#         if (seed + 1) % 5 == 0:
#             print(f"Both intervals: completed {seed + 1}/20 splits")
#     return pd.DataFrame(rows), pd.DataFrame(alpha_rows)

# both_interval_scores, both_interval_alphas = compare_both_intervals()
# both_interval_summary = both_interval_scores.groupby(["test_scope", "model"]).agg(
#     mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
#     mean_participant_MAE_days=("participant_mean_MAE", "mean"),
#     n_splits=("MAE", "size"),
# )
# # Fair paired comparison: SAME Interval 1 test rows, with/without Interval 2 training.
# interval1_training_comparison = both_interval_scores.loc[
#     both_interval_scores.test_scope.eq("Interval 1 only")
# ].merge(repeated_model_scores[["seed", "model", "MAE"]],
#         on=["seed", "model"], suffixes=("_both_training", "_i1_training"), validate="one_to_one")
# interval1_training_comparison["change_days"] = (
#     interval1_training_comparison.MAE_both_training - interval1_training_comparison.MAE_i1_training
# )
# interval1_change_summary = interval1_training_comparison.groupby("model").agg(
#     i1_training_MAE=("MAE_i1_training", "mean"),
#     both_training_MAE=("MAE_both_training", "mean"),
#     change_days=("change_days", "mean"),
#     improved_splits=("change_days", lambda x: int((x < 0).sum())),
# )
# print("Sensitivity analysis: unresolved Interval 2 data quality/label differences.")
# print("Pooled MAE changes the test population; use the same-Interval-1 comparison below.")
# display(both_interval_summary.round(3))
# print("Same Interval 1 test rows: negative change means adding training data helped.")
# display(interval1_change_summary.round(3))
# display(both_interval_alphas.alpha.value_counts().sort_index().rename("n_splits").to_frame())
# # Exploratory overlapping splits, not independent confidence intervals. No exports.


Try ridge regression: learning conditional relationships is good for longitudinal/cyclical data.

In [ ]:
# Random forest: cycle day + history + wearables; Interval 1, same splits as ridge.
# Exploratory repeated evaluation; not an untouched final holdout.
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, GroupKFold
from IPython.display import display

def evaluate_random_forest_6d():
    # Scaling is unnecessary for trees. Imputation is fitted inside each fold.
    pipeline = Pipeline([
        ("impute", SimpleImputer(
            strategy="median", add_indicator=True, keep_empty_features=True,
        )),
        ("forest", RandomForestRegressor(
            n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=-1,
        )),
    ])
    # Small, predefined search: limit depth and require multiple rows per leaf.
    # Leaves contain rows, not necessarily distinct participants.
    parameter_grid = {
        "forest__max_depth": [3, 6],
        "forest__min_samples_leaf": [10, 30],
    }
    score_rows, parameter_rows = [], []
    for seed in range(20):
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(model_frame_6d)
        assert train.study_interval.eq(2022).all() and test.study_interval.eq(2022).all()
        assert set(train.id).isdisjoint(test.id)
        if train.id.nunique() < 2 or test.empty:
            raise ValueError("Insufficient participants for nested evaluation.")
        search = GridSearchCV(
            pipeline, param_grid=parameter_grid,
            scoring="neg_mean_absolute_error",
            cv=GroupKFold(n_splits=min(5, train.id.nunique())),
            refit=True, error_score="raise", n_jobs=1,
        )
        search.fit(train[ridge_feature_columns], train[TARGET], groups=train.id)
        predicted = search.predict(test[ridge_feature_columns])
        assert np.isfinite(predicted).all()
        error = np.abs(test[TARGET].to_numpy() - predicted)
        score_rows.append({
            "seed": seed, "model": "random-forest-6d",
            "MAE": error.mean(),
            "participant_mean_MAE": pd.Series(
                error, index=test.id.to_numpy()
            ).groupby(level=0).mean().mean(),
        })
        parameter_rows.append({
            "seed": seed,
            "max_depth": search.best_params_["forest__max_depth"],
            "min_samples_leaf": search.best_params_["forest__min_samples_leaf"],
        })
        if (seed + 1) % 5 == 0:
            print(f"Random forest: completed {seed + 1}/20 splits")
    return pd.DataFrame(score_rows), pd.DataFrame(parameter_rows)

rf_6d_scores, rf_6d_parameters = evaluate_random_forest_6d()
# Uses the Interval 1 ridge/baseline results, NOT the both-interval sensitivity analysis.
rf_comparison_scores = pd.concat([repeated_model_scores, rf_6d_scores, history_ridge_scores, history_rf_scores], ignore_index=True)
rf_comparison_summary = rf_comparison_scores.groupby("model").agg(
    mean_MAE_days=("MAE", "mean"),
    split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
).sort_values("mean_MAE_days")
rf_paired_mae = rf_comparison_scores.pivot(index="seed", columns="model", values="MAE")
assert rf_paired_mae.shape == (20, 7) and rf_paired_mae.notna().all().all()
rf_paired_comparison = pd.DataFrame([
    {
        "comparison_model": name,
        "mean_RF_minus_comparison_days": (
            rf_paired_mae["random-forest-6d"] - rf_paired_mae[name]
        ).mean(),
        "RF_better_splits": int((rf_paired_mae["random-forest-6d"] < rf_paired_mae[name]).sum()),
        "n_splits": len(rf_paired_mae),
    }
    for name in ("population", "calendar-day", "personalised", "ridge-6d",
                 "ridge-history-only", "random-forest-history-only")
]).set_index("comparison_model")
display(rf_comparison_summary.round(3))
print("Negative paired differences favour random forest. Split SD is not a confidence interval.")
display(rf_paired_comparison.round(3))
display(rf_6d_parameters.groupby(["max_depth", "min_samples_leaf"]).size().rename("n_splits").to_frame())
# No exports: clear outputs before committing. Cleaning/alignment caveats still apply.


In [ ]:
# Separate three-day experiment: history + wearables, Interval 1 only.
# Require three valid calendar days; no latest-day raw-value predictors added.
# The six-day functions, feature lists and results are left unchanged.
def build_three_day_features(daily, prediction_keys):
    # Include prediction dates so windows also work after the final recorded day.
    keys = pd.concat([daily[wearables.KEY], prediction_keys[wearables.KEY]])
    keys = keys.drop_duplicates()
    grid = wearables.dense_days(keys)
    grid = grid.merge(daily, on=wearables.KEY, how="left", validate="one_to_one")
    pieces = []
    for _, person in grid.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        result = person[wearables.KEY].copy()
        for signal in wearables.SIGNALS:
            # HRV night N includes morning N+1. Extra lag avoids prediction-day readings.
            # HRV uses night labels D-4..D-2; wake-day signals use D-3..D-1.
            previous = person[signal].shift(2 if signal == "hrv_rmssd" else 1)
            window = previous.rolling(3, min_periods=3)
            prefix = f"{signal}_3d_"
            result[prefix + "median"] = window.median()
            result[prefix + "sd"] = window.std(ddof=1)
            # Dense calendar grid: x positions correspond to days, including gaps.
            result[prefix + "slope"] = window.apply(
                lambda values: np.polyfit(np.arange(len(values)), values, 1)[0],
                raw=True,
            )
            result[prefix + "min"] = window.min()
            result[prefix + "max"] = window.max()
        pieces.append(result)
    return pd.concat(pieces, ignore_index=True)


three_day_feature_columns = [
    f"{signal}_3d_{stat}" for signal in wearables.SIGNALS
    for stat in ("median", "sd", "slope", "min", "max")
]
three_day_features = build_three_day_features(daily_model_signals, model_frame)
model_frame_3d = history_frame.merge(
    three_day_features[wearables.KEY + three_day_feature_columns],
    on=wearables.KEY, how="left", validate="one_to_one",
)
three_day_model_columns = history_feature_columns + three_day_feature_columns
pd.testing.assert_frame_equal(
    model_frame_3d[wearables.KEY + history_feature_columns + [TARGET]],
    model_frame_6d[wearables.KEY + history_feature_columns + [TARGET]],
)
assert len(three_day_model_columns) == 23

def evaluate_three_day_models():
    from sklearn.base import clone
    from sklearn.ensemble import RandomForestRegressor

    forest = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("forest", RandomForestRegressor(
            n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=-1,
        )),
    ])
    configurations = {
        "ridge-3d": (ridge_pipeline, {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]}),
        "random-forest-3d": (forest, {
            "forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30],
        }),
    }
    rows, parameters = [], []
    for seed in range(20):
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(model_frame_3d)
        assert train.study_interval.eq(2022).all() and test.study_interval.eq(2022).all()
        assert set(train.id).isdisjoint(test.id)
        if train.id.nunique() < 2 or test.empty:
            raise ValueError("Insufficient participants for nested evaluation.")
        for name, (pipeline, grid) in configurations.items():
            search = GridSearchCV(
                clone(pipeline), param_grid=grid, scoring="neg_mean_absolute_error",
                cv=GroupKFold(n_splits=min(5, train.id.nunique())),
                refit=True, error_score="raise", n_jobs=1,
            )
            search.fit(train[three_day_model_columns], train[TARGET], groups=train.id)
            prediction = search.predict(test[three_day_model_columns])
            assert np.isfinite(prediction).all()
            error = np.abs(test[TARGET].to_numpy() - prediction)
            rows.append({
                "seed": seed, "model": name, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
            })
            parameters.append({"seed": seed, "model": name, **search.best_params_})
        if (seed + 1) % 5 == 0:
            print(f"Three-day models: completed {seed + 1}/20 splits")
    return pd.DataFrame(rows), pd.DataFrame(parameters)

three_day_scores, three_day_parameters = evaluate_three_day_models()
window_comparison_scores = pd.concat([
    history_ridge_scores, history_rf_scores,
    repeated_model_scores.loc[repeated_model_scores.model.eq("ridge-6d")],
    rf_6d_scores, three_day_scores,
], ignore_index=True)
window_comparison_summary = window_comparison_scores.groupby("model").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
).sort_values("mean_MAE_days")
window_paired_mae = window_comparison_scores.pivot(index="seed", columns="model", values="MAE")
assert window_paired_mae.shape == (20, 6) and window_paired_mae.notna().all().all()
window_changes = pd.DataFrame([
    {
        "model": model,
        "mean_3d_minus_6d_days": (window_paired_mae[f"{model}-3d"] - window_paired_mae[f"{model}-6d"]).mean(),
        "three_day_better_splits": int((window_paired_mae[f"{model}-3d"] < window_paired_mae[f"{model}-6d"]).sum()),
    }
    for model in ("ridge", "random-forest")
]).set_index("model")
display(window_comparison_summary.round(3))
print("Negative difference favours three days. Same test rows; shorter windows can also reduce missingness.")
display(window_changes.round(3))
# Tuning choices are retained in three_day_parameters. No exports.
# Exploratory overlapping splits; split SD is not a confidence interval.


Results: History still beating History + Wearables. with 3-day windows the closest. Rethink features by researching successful predictive models.

Rise in temperature is known to be associated with an increase in progesterone (increasing during luteal phase). Wrist skin temp, such as those signals from the mcPHASES study, is also strongly influenced by illness, alcohol, and travel as uncovered by [Georgianna et al.](https://pmc.ncbi.nlm.nih.gov/articles/PMC11452339/). This noise is something that wearable data will inevitably carry, so developping features to isolate meaningful change is beneficial.

Try adding 2 features: 1- get 14 day median baseline temperature and get the 3 day median change from baseline. 2- record temperature baseline at period and get change from recorded period temperature. 

Record results from adding column 1, column 2, and then both together.

In [ ]:
# Temperature references: past-only features, Interval 1, original skin-temperature units.
# Fixed choices for this experiment, not tuned against test outcomes.
TEMP_REFERENCE_MIN_READINGS = 7  # At least 7 of the preceding 14 calendar days.
temperature_relative_columns = ["temp_vs_14d_baseline", "temp_vs_period_baseline"]

def add_temperature_references(frame, daily):
    keys = pd.concat([daily[wearables.KEY], frame[wearables.KEY]]).drop_duplicates()
    grid = wearables.dense_days(keys).merge(
        daily[wearables.KEY + ["skin_temp"]],
        on=wearables.KEY, how="left", validate="one_to_one",
    )
    pieces = []
    for _, person in grid.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        temp = person.skin_temp
        result = person[wearables.KEY].copy()
        # For prediction D: recent = D-3..D-1; reference = D-17..D-4.
        result["temp_recent_3d"] = temp.shift(1).rolling(3, min_periods=3).median()
        result["temp_reference_14d"] = temp.shift(4).rolling(
            14, min_periods=TEMP_REFERENCE_MIN_READINGS,
        ).median()
        result["temp_vs_14d_baseline"] = result.temp_recent_3d - result.temp_reference_14d
        pieces.append(result)
    out = frame.merge(pd.concat(pieces, ignore_index=True), on=wearables.KEY,
                      how="left", validate="one_to_one")

    # These are the first three CALENDAR days after onset, not necessarily
    # three confirmed bleeding days. Do not substitute later readings if missing.
    out["period_start_day"] = out.day_in_study - out.cycle_day + 1
    period_keys = ["id", "study_interval", "period_start_day"]
    periods = out[period_keys].drop_duplicates()
    slots = pd.concat([
        periods.assign(day_in_study=periods.period_start_day + offset)
        for offset in range(3)
    ], ignore_index=True)
    slots = slots.merge(daily[wearables.KEY + ["skin_temp"]], on=wearables.KEY,
                        how="left", validate="many_to_one")
    refs = slots.groupby(period_keys, as_index=False).agg(
        temp_reference_period=("skin_temp", "median"),
        period_temp_count=("skin_temp", "count"),
    )
    refs.loc[refs.period_temp_count < 3, "temp_reference_period"] = np.nan
    out = out.merge(refs, on=period_keys, how="left", validate="many_to_one")
    # Even though retrospective data are loaded, never expose days 1..3 early.
    out.loc[out.cycle_day < 4, ["temp_reference_period", "period_temp_count"]] = np.nan
    out["temp_vs_period_baseline"] = out.temp_recent_3d - out.temp_reference_period
    return out

temperature_experiment_frame = add_temperature_references(model_frame_3d, daily_model_signals)
assert len(temperature_experiment_frame) == len(model_frame_3d)
assert temperature_experiment_frame.study_interval.eq(2022).all()
assert temperature_experiment_frame.loc[
    temperature_experiment_frame.cycle_day < 4, "temp_vs_period_baseline"
].isna().all()
# Diagnostic columns (reference values, counts and onset date) are NOT model inputs.
temperature_variants = {
    "neither": [],
    "rolling only": ["temp_vs_14d_baseline"],
    "period only": ["temp_vs_period_baseline"],
    "both": temperature_relative_columns,
}
temperature_train, _ = model_split.apply(temperature_experiment_frame)
display(pd.DataFrame({
    "available_training_rows": temperature_train[temperature_relative_columns].notna().sum(),
    "pct_missing": 100 * temperature_train[temperature_relative_columns].isna().mean(),
}).round(2))


In [ ]:
# Four variants, each with the SAME history + three-day wearable base features.
# Tune within outer training participants; score all variants on identical test rows.
# These repeated test comparisons are exploratory, not an untouched final holdout.
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GridSearchCV, GroupKFold
from IPython.display import display

# Fail before model fitting if prerequisite cells have not run in this kernel.
required_temperature_inputs = [
    "add_temperature_references", "TEMP_REFERENCE_MIN_READINGS",
    "model_frame_3d", "daily_model_signals", "three_day_model_columns",
    "baseline_cycles", "ridge_pipeline", "TARGET", "SEED",
]
missing_temperature_inputs = [name for name in required_temperature_inputs if name not in globals()]
if missing_temperature_inputs:
    raise RuntimeError(
        "Run the preceding 'Temperature references: past-only features' cell first. "
        "If the kernel was restarted, run the setup and three-day feature cells too. "
        f"Missing: {', '.join(missing_temperature_inputs)}"
    )
# Rebuild rather than depending on a possibly absent or stale intermediate table.
temperature_experiment_frame = add_temperature_references(model_frame_3d, daily_model_signals)
temperature_variants = {
    "neither": [],
    "rolling only": ["temp_vs_14d_baseline"],
    "period only": ["temp_vs_period_baseline"],
    "both": ["temp_vs_14d_baseline", "temp_vs_period_baseline"],
}
assert len(temperature_experiment_frame) == len(model_frame_3d)
assert temperature_experiment_frame.study_interval.eq(2022).all()

def evaluate_temperature_variants():
    forest = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("forest", RandomForestRegressor(
            n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=-1,
        )),
    ])
    models = {
        "ridge": (ridge_pipeline, {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]}),
        "random forest": (forest, {
            "forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30],
        }),
    }
    rows, parameters = [], []
    for seed in range(20):
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(temperature_experiment_frame)
        assert set(train.id).isdisjoint(test.id)
        if train.id.nunique() < 2 or test.empty:
            raise ValueError("Insufficient participants for nested evaluation.")
        inner_splits = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
            train, train[TARGET], groups=train.id,
        ))
        for variant, extra_columns in temperature_variants.items():
            columns = three_day_model_columns + extra_columns
            for name, (pipeline, grid) in models.items():
                search = GridSearchCV(
                    clone(pipeline), param_grid=grid, scoring="neg_mean_absolute_error",
                    cv=inner_splits, refit=True, error_score="raise", n_jobs=1,
                )
                search.fit(train[columns], train[TARGET])
                predicted = search.predict(test[columns])
                assert np.isfinite(predicted).all()
                error = np.abs(test[TARGET].to_numpy() - predicted)
                rows.append({
                    "seed": seed, "model": name, "variant": variant,
                    "MAE": error.mean(),
                    "participant_mean_MAE": pd.Series(
                        error, index=test.id.to_numpy()
                    ).groupby(level=0).mean().mean(),
                    "n_test_rows": len(test),
                })
                parameters.append({
                    "seed": seed, "model": name, "variant": variant, **search.best_params_,
                })
        print(f"Temperature variants: completed {seed + 1}/20 splits", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)

temperature_variant_scores, temperature_variant_parameters = evaluate_temperature_variants()
assert len(temperature_variant_scores) == 160
assert temperature_variant_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
temperature_variant_summary = temperature_variant_scores.groupby(["model", "variant"]).agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
)
temperature_paired = temperature_variant_scores.pivot(
    index=["seed", "model"], columns="variant", values="MAE",
)
assert temperature_paired.notna().all().all()
temperature_differences = temperature_paired.subtract(temperature_paired["neither"], axis=0)
temperature_change_summary = temperature_differences.groupby(level="model").mean()
print("Mean MAE in days (lower is better):")
display(temperature_variant_summary.mean_MAE_days.unstack("variant").reindex(
    columns=list(temperature_variants)
).round(3))
print("Change versus neither: negative means improvement on the same test rows.")
display(temperature_change_summary.reindex(columns=list(temperature_variants)).round(3))
print("Splits improved versus neither (out of 20; not a significance test):")
display(temperature_differences.lt(0).groupby(level="model").sum().reindex(
    columns=list(temperature_variants)
))
# Detailed split SD and participant-weighted MAE remain in temperature_variant_summary.
# Missingness indicators are included by the pipelines: improvements may reflect
# availability as well as temperature values. No exports; clear outputs before committing.


Minimal difference, but feature 1 performs better on its own. Try different windows. from 7 - 20 day.

In [ ]:
# Window search on TRAINING participants only (not the 20 outer-test MAEs).
# History + three-day wearables stay fixed; add ONE rolling-temperature deviation.
# Recent window: D-3..D-1. Reference of W days: D-(W+3)..D-4.
# Require ceil(W/2) valid reference readings; all three recent days must be valid.
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from IPython.display import display

temperature_window_lengths = list(range(7, 21))

def build_temperature_window_candidates(daily, frame, lengths):
    keys = pd.concat([daily[wearables.KEY], frame[wearables.KEY]]).drop_duplicates()
    grid = wearables.dense_days(keys).merge(
        daily[wearables.KEY + ["skin_temp"]], on=wearables.KEY,
        how="left", validate="one_to_one",
    )
    pieces = []
    for _, person in grid.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        temp = person.skin_temp
        result = person[wearables.KEY].copy()
        recent = temp.shift(1).rolling(3, min_periods=3).median()
        for days in lengths:
            reference = temp.shift(4).rolling(days, min_periods=(days + 1) // 2).median()
            result[f"temp_vs_{days}d_reference"] = recent - reference
        pieces.append(result)
    return frame.merge(pd.concat(pieces, ignore_index=True), on=wearables.KEY,
                       how="left", validate="one_to_one")

temperature_window_frame = build_temperature_window_candidates(
    daily_model_signals, model_frame_3d, temperature_window_lengths,
)
temperature_window_train, _ = model_split.apply(temperature_window_frame)
assert temperature_window_train.study_interval.eq(2022).all()
assert set(temperature_window_train.id).isdisjoint(model_split.test_ids)

def search_temperature_window_lengths():
    train = temperature_window_train
    folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
        train, train[TARGET], groups=train.id,
    ))
    for fitting, validation in folds:
        assert set(train.id.iloc[fitting]).isdisjoint(train.id.iloc[validation])
    forest = Pipeline([
        ("impute", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True)),
        ("forest", RandomForestRegressor(
            n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=-1,
        )),
    ])
    models = {
        "ridge": (ridge_pipeline, {"ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]}),
        "random forest": (forest, {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}),
    }
    rows, candidates = [], []
    # Recompute the no-added-feature control on these SAME validation folds.
    for days in [None] + temperature_window_lengths:
        extra = [] if days is None else [f"temp_vs_{days}d_reference"]
        columns = three_day_model_columns + extra
        missing_pct = np.nan if days is None else 100 * train[extra[0]].isna().mean()
        for name, (pipeline, grid) in models.items():
            search = GridSearchCV(
                clone(pipeline), param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=False, error_score="raise", n_jobs=1,
            )
            search.fit(train[columns], train[TARGET])
            best = search.best_index_
            rows.append({
                "model": name, "reference_days": 0 if days is None else days,
                "validation_MAE_days": -search.best_score_,
                "fold_SD_days": search.cv_results_["std_test_score"][best],
                "reference_feature_pct_missing": missing_pct,
                "best_parameters": search.best_params_,
            })
            for i, params in enumerate(search.cv_results_["params"]):
                candidates.append({
                    "model": name, "reference_days": 0 if days is None else days,
                    "parameters": params, "validation_MAE_days": -search.cv_results_["mean_test_score"][i],
                })
        print(f"Completed reference window: {days if days is not None else 'no added feature'}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(candidates)

temperature_window_results, temperature_window_candidates = search_temperature_window_lengths()
control = temperature_window_results.loc[
    temperature_window_results.reference_days.eq(0)
].set_index("model").validation_MAE_days
temperature_window_results["change_vs_no_added_feature_days"] = (
    temperature_window_results.validation_MAE_days - temperature_window_results.model.map(control)
)
print("Training-participant validation MAE. Reference 0 = no added temperature-relative feature.")
display(temperature_window_results.pivot(
    index="reference_days", columns="model", values="validation_MAE_days",
).round(3))
print("Best reference window per model (excluding the no-added-feature control):")
temperature_best_windows = (
    temperature_window_results.loc[temperature_window_results.reference_days.gt(0)]
    .sort_values(["validation_MAE_days", "reference_days"])
    .groupby("model", sort=False).head(1)
)
display(temperature_best_windows.round(3))
print("These are window/parameter selection scores, not independent test estimates.")
# Per-window missingness and SD are retained in temperature_window_results.
# Minimum-reading counts vary with window length, so coverage also changes.
# No full-study normalization, period reference, new exports, or test-set scoring.


No meaningful difference. Re-investigate data to see if it is malformed. To do this, let's start with history, then add wearable data column 1 at a time, to see if there is a column which may be adding unneccessary noise.

In [ ]:
# One-at-a-time additions to history-only RF; Interval 1, same 20 outer splits.
# Each variant adds ONE numeric feature (+ its missingness indicator if needed).
# No cumulative additions, period-temperature reference, or Interval 2 data.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

single_feature_candidates = three_day_feature_columns + ["temp_vs_14d_baseline"]
single_feature_frame = model_frame_3d.merge(
    temperature_experiment_frame[wearables.KEY + ["temp_vs_14d_baseline"]],
    on=wearables.KEY, how="left", validate="one_to_one",
)
assert len(single_feature_frame) == len(history_frame)
assert single_feature_frame.study_interval.eq(2022).all()
assert len(single_feature_candidates) == 21
pd.testing.assert_frame_equal(
    single_feature_frame[wearables.KEY + history_feature_columns + [TARGET]],
    history_frame[wearables.KEY + history_feature_columns + [TARGET]],
)

def evaluate_single_feature_forests():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for feature in [None] + single_feature_candidates:
        label = feature or "history_only"
        columns = history_feature_columns + ([] if feature is None else [feature])
        transforms = [("history", history_imputer, history_feature_columns)]
        if feature is not None:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True,
            ), [feature]))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(single_feature_frame)
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "test_feature_pct_missing": (
                    np.nan if feature is None else 100 * test[feature].isna().mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)

single_feature_scores, single_feature_parameters = evaluate_single_feature_forests()
assert len(single_feature_scores) == 22 * 20
assert single_feature_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
single_feature_control = single_feature_scores.loc[
    single_feature_scores.feature.eq("history_only"), ["seed", "MAE"]
].rename(columns={"MAE": "history_MAE"})
single_feature_paired = single_feature_scores.merge(
    single_feature_control, on="seed", how="left", validate="many_to_one",
)
single_feature_paired["MAE_change"] = single_feature_paired.MAE - single_feature_paired.history_MAE
single_feature_summary = single_feature_paired.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), change_vs_history_days=("MAE_change", "mean"),
    improved_splits=("MAE_change", lambda x: int((x < 0).sum())),
    split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_missing=("test_feature_pct_missing", "mean"),
).sort_values("mean_MAE_days")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(single_feature_summary.round(3))
print("Negative change favours the added feature; improved_splits is out of 20.")
# Exploratory overlapping tests: not independent experiments or significance tests.
# A worse MAE flags a candidate for investigation, not proof of a pipeline defect.
# Scores include rows with missing features, imputed inside training folds only.
# No exports; clear notebook outputs before committing.


Try latest day values instead of 3-day window. Start with breathing rate column and see if it makes a difference.

In [ ]:
# History + previous day's breathing-rate daily summary (NOT latest nonmissing).
# No forward-filling: an absent D-1 reading stays missing, even if D-2 exists.
# Interval 1, original 20 participant splits, nested tuning unchanged.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def previous_day_breathing(daily):
    previous = daily[wearables.KEY + ["breathing_rate"]].copy()
    previous["day_in_study"] = previous.day_in_study + 1
    return previous.rename(columns={"breathing_rate": "breathing_rate_previous_day"})

breathing_comparison_frame = model_frame_3d.merge(
    previous_day_breathing(daily_model_signals),
    on=wearables.KEY, how="left", validate="one_to_one",
)
assert breathing_comparison_frame.study_interval.eq(2022).all()
assert len(breathing_comparison_frame) == len(history_frame)
# Keep only history + ONE breathing feature in each model; not all wearables.
def evaluate_breathing_rate_variants():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for feature in [None] + ["breathing_rate_previous_day", "breathing_rate_3d_median"]:
        label = feature or "history_only"
        columns = history_feature_columns + ([] if feature is None else [feature])
        transforms = [("history", history_imputer, history_feature_columns)]
        if feature is not None:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True,
            ), [feature]))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(breathing_comparison_frame)
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "test_feature_pct_missing": (
                    np.nan if feature is None else 100 * test[feature].isna().mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)


breathing_comparison_scores, breathing_comparison_parameters = evaluate_breathing_rate_variants()
assert len(breathing_comparison_scores) == 60
assert breathing_comparison_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
breathing_mae_paired = breathing_comparison_scores.pivot(index="seed", columns="feature", values="MAE")
breathing_changes = breathing_mae_paired.subtract(breathing_mae_paired.history_only, axis=0)
breathing_comparison_summary = breathing_comparison_scores.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_missing=("test_feature_pct_missing", "mean"),
)
breathing_comparison_summary["change_vs_history_days"] = breathing_changes.mean()
breathing_comparison_summary["better_than_history_splits"] = breathing_changes.lt(0).sum()
with pd.option_context("display.max_columns", None):
    display(breathing_comparison_summary.round(3))
print("Previous-day reading beats the three-day median in",
      int((breathing_mae_paired.breathing_rate_previous_day <
           breathing_mae_paired.breathing_rate_3d_median).sum()), "of 20 splits.")
# Exploratory overlapping splits. Missingness indicators may contribute to performance.
# No files exported; clear outputs before committing.


Investigate temperature for random pariicipants. I'm suprised to see that it doesn't have a positive impact on the model

In [ ]:
# Inspect the exact 14-day-reference feature used by the models (before imputation).
# Three random TRAINING participants; selection does not depend on model performance.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if "temperature_experiment_frame" not in globals():
    raise RuntimeError("Run the 'Temperature references: past-only features' cell first.")
temperature_plot_frame, _ = model_split.apply(temperature_experiment_frame)
temperature_plot_frame = temperature_plot_frame.loc[
    temperature_plot_frame.study_interval.eq(2022)
].copy()
temperature_plot_ids = np.sort(temperature_plot_frame.id.unique())
if len(temperature_plot_ids) < 3:
    raise ValueError("At least three training participants are needed.")
temperature_plot_ids = np.random.default_rng(SEED).choice(
    temperature_plot_ids, size=3, replace=False,
)
assert set(temperature_plot_ids).issubset(set(model_split.train_ids))

temperature_plot_data = {}
temperature_plot_fig, temperature_plot_axes = plt.subplots(
    3, 2, figsize=(15, 10), sharey="col", layout="constrained",
)
for row, participant in enumerate(temperature_plot_ids):
    person = temperature_plot_frame.loc[
        temperature_plot_frame.id.eq(participant)
    ].sort_values("day_in_study").set_index("day_in_study")
    # Preserve chronological gaps instead of drawing across absent prediction rows.
    dates = np.arange(int(person.index.min()), int(person.index.max()) + 1)
    person = person.reindex(dates)
    temperature_plot_data[participant] = person
    levels_ax, change_ax = temperature_plot_axes[row]
    levels_ax.plot(dates, person.temp_recent_3d, color="C0", linewidth=1.6,
                   marker=".", markersize=3, label="Recent 3-day median")
    levels_ax.plot(dates, person.temp_reference_14d, color="C1", linestyle="--",
                   linewidth=1.6, label="Earlier 14-day reference")
    change_ax.plot(dates, person.temp_vs_14d_baseline, color="C2", linewidth=1.6,
                   marker=".", markersize=3, label="Recent minus reference")
    change_ax.axhline(0, color="0.5", linewidth=0.8)

    # Mark feature NaNs ONLY on actual model rows; gaps outside model rows stay blank.
    missing = person.cycle_day.notna() & person.temp_vs_14d_baseline.isna()
    change_ax.scatter(dates[missing], np.full(int(missing.sum()), 0.025),
                      transform=change_ax.get_xaxis_transform(),
                      marker="|", color="0.5", s=45, label="Feature unavailable")
    period_days = baseline_starts.loc[
        baseline_starts.id.eq(participant) & baseline_starts.study_interval.eq(2022),
        "start_day",
    ]
    for ax in (levels_ax, change_ax):
        for i, day in enumerate(period_days.loc[period_days.between(dates[0], dates[-1])]):
            ax.axvline(day, color="C3", linestyle=":", linewidth=1, alpha=0.7,
                       label="Recorded period start" if i == 0 else None)
        ax.set_xlim(dates[0] - 0.5, dates[-1] + 0.5)
        ax.set_xlabel("Prediction day in study (not cycle day)")
        ax.grid(True, alpha=0.2)
    levels_ax.set_ylabel("Skin temperature (°C)")
    change_ax.set_ylabel("Temperature difference (°C)")
    levels_ax.set_title(f"Participant {participant} — recent and reference temperatures", loc="left")
    available = person.temp_vs_14d_baseline.notna().sum()
    model_days = person.cycle_day.notna().sum()
    change_ax.set_title(f"Model feature — available on {available}/{model_days} prediction days", loc="left")
    if row == 0:
        levels_ax.legend(fontsize=8, loc="best")
        change_ax.legend(fontsize=8, loc="best")

temperature_plot_fig.suptitle(
    "Interval 1: three random training participants — temperature change relative to earlier readings\n"
    "For prediction D: recent D−3…D−1 (3 valid days); reference D−17…D−4 (at least 7 valid days)",
    fontsize=12,
)
plt.show()
# Positive values mean warmer than the earlier reference; zero means equal medians.
# Gaps are NOT imputed. Period markers are context, not proof of ovulation.
# Plot is exploratory; no figure exported. Clear participant outputs before committing.


In [ ]:
# Compare candidate temperature inputs at prediction day D, before imputation.
# Latest = D-1 only (no carry-forward); medians use D-3..D-1 and D-6..D-1.
# Reproduce the same random training-participant selection as the preceding plot.
skin_input_training, _ = model_split.apply(model_frame_3d)
skin_input_training = skin_input_training.loc[skin_input_training.study_interval.eq(2022)]
skin_input_ids = np.sort(skin_input_training.id.unique())
if len(skin_input_ids) < 3:
    raise ValueError("At least three training participants are needed.")
skin_input_ids = np.random.default_rng(SEED).choice(skin_input_ids, 3, replace=False)
assert set(skin_input_ids).issubset(set(model_split.train_ids))

skin_previous = daily_model_signals[wearables.KEY + ["skin_temp"]].copy()
skin_previous["day_in_study"] = skin_previous.day_in_study + 1
skin_previous = skin_previous.rename(columns={"skin_temp": "skin_temp_previous_day"})
skin_input_frame = skin_input_training[wearables.KEY + ["skin_temp_3d_median"]].merge(
    model_frame_6d[wearables.KEY + ["skin_temp_6d_median"]],
    on=wearables.KEY, how="left", validate="one_to_one",
).merge(skin_previous, on=wearables.KEY, how="left", validate="one_to_one")

skin_input_plot_data = {}
skin_input_fig, skin_input_axes = plt.subplots(
    3, 1, figsize=(14, 10), sharey=True, layout="constrained",
)
for ax, participant in zip(skin_input_axes, skin_input_ids):
    person = skin_input_frame.loc[skin_input_frame.id.eq(participant)].set_index("day_in_study").sort_index()
    dates = np.arange(int(person.index.min()), int(person.index.max()) + 1)
    person = person.reindex(dates)  # Do not bridge missing days or gaps between model rows.
    skin_input_plot_data[participant] = person
    ax.plot(dates, person.skin_temp_previous_day, color="0.55", linewidth=1,
            marker=".", markersize=3, alpha=0.75, label="Previous-day skin temperature")
    ax.plot(dates, person.skin_temp_3d_median, color="C0", linewidth=1.8,
            linestyle="-", label="3-day median")
    ax.plot(dates, person.skin_temp_6d_median, color="C1", linewidth=1.8,
            linestyle="--", label="6-day median")
    starts = baseline_starts.loc[
        baseline_starts.id.eq(participant) & baseline_starts.study_interval.eq(2022),
        "start_day",
    ]
    for i, day in enumerate(starts.loc[starts.between(dates[0], dates[-1])]):
        ax.axvline(day, color="C3", linestyle=":", linewidth=1.2,
                   label="Recorded period start" if i == 0 else None)
    ax.set_title(f"Participant {participant} — Interval 1, training", loc="left")
    ax.set_ylabel("Skin temperature (°C)")
    ax.set_xlabel("Prediction day in study (not cycle day)")
    ax.set_xlim(dates[0] - 0.5, dates[-1] + 0.5)
    ax.grid(True, alpha=0.2)
    if ax is skin_input_axes[0]:
        ax.legend(ncol=2, fontsize=9, loc="best")
skin_input_fig.suptitle(
    "Temperature inputs available before prediction day\n"
    "Latest: D−1 | 3-day median: D−3…D−1 | 6-day median: D−6…D−1; complete windows required",
    fontsize=12,
)
plt.show()
# The previous-day temperature is plotted as a candidate, not added to the models.
# No interpolation, imputation, or file export. Clear participant outputs before committing.


In [ ]:
# Compare candidate breathing-rate inputs at prediction day D, before imputation.
# Latest = D-1 only (no carry-forward); medians use D-3..D-1 and D-6..D-1.
# Reproduce the same random training-participant selection as the preceding plot.
breathing_input_training, _ = model_split.apply(model_frame_3d)
breathing_input_training = breathing_input_training.loc[breathing_input_training.study_interval.eq(2022)]
breathing_input_ids = np.sort(breathing_input_training.id.unique())
if len(breathing_input_ids) < 3:
    raise ValueError("At least three training participants are needed.")
breathing_input_ids = np.random.default_rng(SEED).choice(breathing_input_ids, 3, replace=False)
assert set(breathing_input_ids).issubset(set(model_split.train_ids))

breathing_previous = daily_model_signals[wearables.KEY + ["breathing_rate"]].copy()
breathing_previous["day_in_study"] = breathing_previous.day_in_study + 1
breathing_previous = breathing_previous.rename(columns={"breathing_rate": "breathing_rate_previous_day"})
breathing_input_frame = breathing_input_training[wearables.KEY + ["breathing_rate_3d_median", "breathing_rate_3d_min", "breathing_rate_3d_max"]].merge(
    model_frame_6d[wearables.KEY + ["breathing_rate_6d_median"]],
    on=wearables.KEY, how="left", validate="one_to_one",
).merge(breathing_previous, on=wearables.KEY, how="left", validate="one_to_one")

breathing_input_plot_data = {}
breathing_input_fig, breathing_input_axes = plt.subplots(
    3, 1, figsize=(14, 10), sharey=True, layout="constrained",
)
for ax, participant in zip(breathing_input_axes, breathing_input_ids):
    person = breathing_input_frame.loc[breathing_input_frame.id.eq(participant)].set_index("day_in_study").sort_index()
    dates = np.arange(int(person.index.min()), int(person.index.max()) + 1)
    person = person.reindex(dates)  # Do not bridge missing days or gaps between model rows.
    breathing_input_plot_data[participant] = person
    ax.plot(dates, person.breathing_rate_previous_day, color="0.55", linewidth=1,
            marker=".", markersize=3, alpha=0.75, label="Previous-day breathing rate")
    ax.plot(dates, person.breathing_rate_3d_median, color="C0", linewidth=1.8,
            linestyle="-", label="3-day median")
    ax.plot(dates, person.breathing_rate_6d_median, color="C1", linewidth=1.8,
            linestyle="--", label="6-day median")
    ax.plot(dates, person.breathing_rate_3d_min, color="C2", linewidth=1.2,
            linestyle="-.", label="3-day minimum")
    ax.plot(dates, person.breathing_rate_3d_max, color="C4", linewidth=1.2,
            linestyle=":", label="3-day maximum")
    starts = baseline_starts.loc[
        baseline_starts.id.eq(participant) & baseline_starts.study_interval.eq(2022),
        "start_day",
    ]
    for i, day in enumerate(starts.loc[starts.between(dates[0], dates[-1])]):
        ax.axvline(day, color="C3", linestyle=":", linewidth=1.2,
                   label="Recorded period start" if i == 0 else None)
    ax.set_title(f"Participant {participant} — Interval 1, training", loc="left")
    ax.set_ylabel("Breathing rate (breaths/min)")
    ax.set_xlabel("Prediction day in study (not cycle day)")
    ax.set_xlim(dates[0] - 0.5, dates[-1] + 0.5)
    ax.grid(True, alpha=0.2)
    if ax is breathing_input_axes[0]:
        ax.legend(ncol=3, fontsize=9, loc="best")
breathing_input_fig.suptitle(
    "Breathing-rate inputs available before prediction day\n"
    "Latest: D−1 | 3-day statistics: D−3…D−1 | 6-day median: D−6…D−1; complete windows required",
    fontsize=12,
)
plt.show()
# Previous-day and 3-day min/median/max improved exploratory MAE; 6-day median is context.
# No interpolation, imputation, or file export. Clear participant outputs before committing.


In [ ]:
# Plot first, then evaluate history + one temperature feature at a time.
# "Latest" matches the earlier plot: D-1 only, NOT forward-filled.
# Keep the existing reference D-17..D-4 (>=7 valid readings) unchanged.
# Interval 1 only. Neither feature uses prediction-day or future measurements.
temperature_previous = daily_model_signals[wearables.KEY + ["skin_temp"]].copy()
temperature_previous["day_in_study"] += 1
temperature_previous = temperature_previous.rename(columns={"skin_temp": "skin_temp_previous_day"})
latest_temperature_frame = temperature_experiment_frame.merge(
    temperature_previous, on=wearables.KEY, how="left", validate="one_to_one",
)
latest_temperature_frame["temp_previous_vs_14d"] = (
    latest_temperature_frame.skin_temp_previous_day - latest_temperature_frame.temp_reference_14d
)
assert len(latest_temperature_frame) == len(history_frame)
assert latest_temperature_frame.study_interval.eq(2022).all()
latest_temperature_training, _ = model_split.apply(latest_temperature_frame)
latest_temperature_ids = np.random.default_rng(SEED).choice(
    np.sort(latest_temperature_training.id.unique()), 3, replace=False,
)
latest_temperature_fig, latest_temperature_axes = plt.subplots(
    3, 2, figsize=(15, 10), sharey="col", layout="constrained",
)
for axes, participant in zip(latest_temperature_axes, latest_temperature_ids):
    person = latest_temperature_training.loc[latest_temperature_training.id.eq(participant)].set_index("day_in_study").sort_index()
    days = np.arange(int(person.index.min()), int(person.index.max()) + 1)
    person = person.reindex(days)
    axes[0].plot(days, person.skin_temp_previous_day, color="C0", marker=".", markersize=3, label="Previous-day temperature")
    axes[0].plot(days, person.temp_reference_14d, color="C1", linestyle="--", label="14-day reference median")
    axes[1].plot(days, person.temp_previous_vs_14d, color="C2", marker=".", markersize=3, label="Previous day minus reference")
    axes[1].axhline(0, color="0.5", linewidth=0.8)
    starts = baseline_starts.loc[baseline_starts.id.eq(participant) & baseline_starts.study_interval.eq(2022), "start_day"]
    for ax in axes:
        for i, day in enumerate(starts.loc[starts.between(days[0], days[-1])]):
            ax.axvline(day, color="C3", linestyle=":", label="Period start" if i == 0 else None)
        ax.set_title(f"Participant {participant} — Interval 1, training", loc="left")
        ax.set_xlabel("Prediction day in study")
        ax.set_xlim(days[0] - 0.5, days[-1] + 0.5)
    axes[0].set_ylabel("Skin temperature (°C)")
    axes[1].set_ylabel("Difference (°C)")
for ax in latest_temperature_axes[0]:
    ax.legend(fontsize=8)
latest_temperature_fig.suptitle("Previous-day temperature and its deviation from the existing 14-day baseline")
plt.show()

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def evaluate_latest_temperature_variants():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for feature in [None] + ["skin_temp_previous_day", "temp_previous_vs_14d"]:
        label = feature or "history_only"
        columns = history_feature_columns + ([] if feature is None else [feature])
        transforms = [("history", history_imputer, history_feature_columns)]
        if feature is not None:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True,
            ), [feature]))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(latest_temperature_frame)
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "test_feature_pct_missing": (
                    np.nan if feature is None else 100 * test[feature].isna().mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)


latest_temperature_comparison_scores, latest_temperature_comparison_parameters = evaluate_latest_temperature_variants()
assert len(latest_temperature_comparison_scores) == 60
assert latest_temperature_comparison_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
latest_temperature_mae_paired = latest_temperature_comparison_scores.pivot(index="seed", columns="feature", values="MAE")
latest_temperature_changes = latest_temperature_mae_paired.subtract(latest_temperature_mae_paired.history_only, axis=0)
latest_temperature_comparison_summary = latest_temperature_comparison_scores.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_missing=("test_feature_pct_missing", "mean"),
)
latest_temperature_comparison_summary["change_vs_history_days"] = latest_temperature_changes.mean()
latest_temperature_comparison_summary["better_than_history_splits"] = latest_temperature_changes.lt(0).sum()
with pd.option_context("display.max_columns", None):
    display(latest_temperature_comparison_summary.round(3))
# Exploratory overlapping splits, not independent significance tests.
# Each wearable model also includes its missingness indicator. No files exported.


In [ ]:
# Previous-day temperature minus a 5–14-calendar-day baseline median.
# Interval 1; history + ONE deviation column per model, not all windows together.
# Keep the prior reference endpoint D-4: window W spans D-(W+3)..D-4.
# D-3..D-1 remain outside the reference; latest reading is D-1, no forward-fill.
# Require >=ceil(W/2) valid baseline days (14 days still requires 7).
temperature_sweep_windows = list(range(5, 15))
temperature_sweep_columns = [f"temp_previous_vs_{w}d" for w in temperature_sweep_windows]

def build_previous_temperature_windows(daily, frame, windows):
    keys = pd.concat([daily[wearables.KEY], frame[wearables.KEY]]).drop_duplicates()
    dense = wearables.dense_days(keys).merge(
        daily[wearables.KEY + ["skin_temp"]], on=wearables.KEY,
        how="left", validate="one_to_one",
    )
    pieces = []
    for _, person in dense.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        result = person[wearables.KEY].copy()
        for window in windows:
            reference = person.skin_temp.shift(4).rolling(
                window, min_periods=int(np.ceil(window / 2)),
            ).median()
            result[f"temp_previous_vs_{window}d"] = person.skin_temp.shift(1) - reference
        pieces.append(result)
    return frame.merge(pd.concat(pieces, ignore_index=True), on=wearables.KEY,
                       how="left", validate="one_to_one")

temperature_sweep_frame = build_previous_temperature_windows(
    daily_model_signals, history_frame, temperature_sweep_windows,
)
assert temperature_sweep_frame.study_interval.eq(2022).all()
assert len(temperature_sweep_frame) == len(history_frame)
# No test rows are dropped because a temperature or baseline is missing.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def evaluate_temperature_window_sweep():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for feature in [None] + temperature_sweep_columns:
        label = feature or "history_only"
        columns = history_feature_columns + ([] if feature is None else [feature])
        transforms = [("history", history_imputer, history_feature_columns)]
        if feature is not None:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True,
            ), [feature]))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(temperature_sweep_frame)
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "test_feature_pct_missing": (
                    np.nan if feature is None else 100 * test[feature].isna().mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)


temperature_sweep_scores, temperature_sweep_parameters = evaluate_temperature_window_sweep()
assert len(temperature_sweep_scores) == 20 * (1 + len(temperature_sweep_windows))
assert temperature_sweep_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
temperature_sweep_mae_paired = temperature_sweep_scores.pivot(index="seed", columns="feature", values="MAE")
temperature_sweep_changes = temperature_sweep_mae_paired.subtract(temperature_sweep_mae_paired.history_only, axis=0)
temperature_sweep_summary = temperature_sweep_scores.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_missing=("test_feature_pct_missing", "mean"),
)
temperature_sweep_summary["change_vs_history_days"] = temperature_sweep_changes.mean()
temperature_sweep_summary["better_than_history_splits"] = temperature_sweep_changes.lt(0).sum()
temperature_sweep_summary = temperature_sweep_summary.sort_values("mean_MAE_days")
with pd.option_context("display.max_columns", None):
    display(temperature_sweep_summary.round(3))
# Exploratory overlapping splits, not independent significance tests.
# Each wearable model also includes its missingness indicator. No files exported.

best_temperature_window_feature = temperature_sweep_summary.loc[
    temperature_sweep_columns, "mean_MAE_days"
].idxmin()
print("Lowest-MAE temperature window:", best_temperature_window_feature,
      "— MAE", round(temperature_sweep_summary.loc[best_temperature_window_feature, "mean_MAE_days"], 4), "days")
print("Best overall (including history only):", temperature_sweep_summary.index[0])
print("Exploratory window selection on repeated test splits; validate the selected window on fresh data.")
# Coverage changes with window length; the table includes missingness for interpretation.


In [ ]:
# Previous-day temperature minus a 5–14-calendar-day baseline median.
# Intervals 1 and 2; history + ONE deviation column per model, not all windows together.
# Keep the prior reference endpoint D-4: window W spans D-(W+3)..D-4.
# D-3..D-1 remain outside the reference; latest reading is D-1, no forward-fill.
# Require >=ceil(W/2) valid baseline days (14 days still requires 7).
both_temperature_sweep_windows = list(range(5, 15))
both_temperature_sweep_columns = [f"temp_previous_vs_{w}d" for w in both_temperature_sweep_windows]

def build_both_previous_temperature_windows(daily, frame, windows):
    keys = pd.concat([daily[wearables.KEY], frame[wearables.KEY]]).drop_duplicates()
    dense = wearables.dense_days(keys).merge(
        daily[wearables.KEY + ["skin_temp"]], on=wearables.KEY,
        how="left", validate="one_to_one",
    )
    pieces = []
    for _, person in dense.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        result = person[wearables.KEY].copy()
        for window in windows:
            reference = person.skin_temp.shift(4).rolling(
                window, min_periods=int(np.ceil(window / 2)),
            ).median()
            result[f"temp_previous_vs_{window}d"] = person.skin_temp.shift(1) - reference
        pieces.append(result)
    return frame.merge(pd.concat(pieces, ignore_index=True), on=wearables.KEY,
                       how="left", validate="one_to_one")

# TODO(INTERVAL2_REVIEW): Sensitivity analysis pending study-label/data-quality review.
# No HR is a predictor here. Keep cycle history and temperature windows within interval.
# Split on participant ID: both intervals of a person always stay in the same fold.
both_temperature_cycles = baseline_cycles.loc[baseline_cycles.study_interval.isin([2022, 2024])].copy()
both_temperature_history = add_prior_cycle_history(
    data.make_daily_prediction_frame(both_temperature_cycles), both_temperature_cycles,
)
both_temperature_daily = wearables.load_daily_signals()
both_temperature_daily = both_temperature_daily.loc[
    both_temperature_daily.study_interval.isin([2022, 2024])
].copy()

both_temperature_sweep_frame = build_both_previous_temperature_windows(
    both_temperature_daily, both_temperature_history, both_temperature_sweep_windows,
)
assert set(both_temperature_sweep_frame.study_interval.unique()) == {2022, 2024}
assert len(both_temperature_sweep_frame) == len(both_temperature_history)
print("Evaluation pools Interval 1 and 2 prediction rows; compare against the pooled history-only control.")
# No test rows are dropped because a temperature or baseline is missing.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def evaluate_temperature_window_sweep():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for feature in [None] + both_temperature_sweep_columns:
        label = feature or "history_only"
        columns = history_feature_columns + ([] if feature is None else [feature])
        transforms = [("history", history_imputer, history_feature_columns)]
        if feature is not None:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True,
            ), [feature]))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(both_temperature_sweep_frame)
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "test_feature_pct_missing": (
                    np.nan if feature is None else 100 * test[feature].isna().mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)


both_temperature_sweep_scores, both_temperature_sweep_parameters = evaluate_temperature_window_sweep()
assert len(both_temperature_sweep_scores) == 20 * (1 + len(both_temperature_sweep_windows))
assert both_temperature_sweep_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
both_temperature_sweep_mae_paired = both_temperature_sweep_scores.pivot(index="seed", columns="feature", values="MAE")
both_temperature_sweep_changes = both_temperature_sweep_mae_paired.subtract(both_temperature_sweep_mae_paired.history_only, axis=0)
both_temperature_sweep_summary = both_temperature_sweep_scores.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_missing=("test_feature_pct_missing", "mean"),
)
both_temperature_sweep_summary["change_vs_history_days"] = both_temperature_sweep_changes.mean()
both_temperature_sweep_summary["better_than_history_splits"] = both_temperature_sweep_changes.lt(0).sum()
both_temperature_sweep_summary = both_temperature_sweep_summary.sort_values("mean_MAE_days")
with pd.option_context("display.max_columns", None):
    display(both_temperature_sweep_summary.round(3))
# Exploratory overlapping splits, not independent significance tests.
# Each wearable model also includes its missingness indicator. No files exported.

best_both_temperature_window_feature = both_temperature_sweep_summary.loc[
    both_temperature_sweep_columns, "mean_MAE_days"
].idxmin()
print("Lowest-MAE temperature window:", best_both_temperature_window_feature,
      "— MAE", round(both_temperature_sweep_summary.loc[best_both_temperature_window_feature, "mean_MAE_days"], 4), "days")
print("Best overall (including history only):", both_temperature_sweep_summary.index[0])
print("Exploratory window selection on repeated test splits; validate the selected window on fresh data.")
# Coverage changes with window length; the table includes missingness for interpretation.


In [ ]:
# Both intervals: history + breathing-rate features + 7-day temperature deviation.
# TODO(INTERVAL2_REVIEW): Exploratory sensitivity analysis pending dataset review.
# Latest = D-1 only, no carry-forward. All features are past-only.
breathing_temperature_cycles = baseline_cycles.loc[baseline_cycles.study_interval.isin([2022, 2024])].copy()
breathing_temperature_history = add_prior_cycle_history(
    data.make_daily_prediction_frame(breathing_temperature_cycles), breathing_temperature_cycles,
)
breathing_temperature_daily = wearables.load_daily_signals()
breathing_temperature_daily = breathing_temperature_daily.loc[
    breathing_temperature_daily.study_interval.isin([2022, 2024])
].copy()
breathing_temperature_columns = [
    "breathing_rate_previous_day", "breathing_rate_3d_min",
    "breathing_rate_3d_median", "breathing_rate_3d_max",
]

def build_breathing_temperature_features(daily, frame):
    keys = pd.concat([daily[wearables.KEY], frame[wearables.KEY]]).drop_duplicates()
    dense = wearables.dense_days(keys).merge(
        daily[wearables.KEY + ["skin_temp", "breathing_rate"]],
        on=wearables.KEY, how="left", validate="one_to_one",
    )
    pieces = []
    for _, person in dense.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        result = person[wearables.KEY].copy()
        previous = person.breathing_rate.shift(1)
        result["breathing_rate_previous_day"] = previous
        window = previous.rolling(3, min_periods=3)  # D-3..D-1, complete calendar window.
        result["breathing_rate_3d_min"] = window.min()
        result["breathing_rate_3d_median"] = window.median()
        result["breathing_rate_3d_max"] = window.max()
        # Preserve the tested baseline: D-10..D-4, at least 4 valid days out of 7.
        result["temp_previous_vs_7d"] = (
            person.skin_temp.shift(1) - person.skin_temp.shift(4).rolling(7, min_periods=4).median()
        )
        pieces.append(result)
    return frame.merge(pd.concat(pieces, ignore_index=True), on=wearables.KEY,
                       how="left", validate="one_to_one")

breathing_temperature_frame = build_breathing_temperature_features(
    breathing_temperature_daily, breathing_temperature_history,
)
assert len(breathing_temperature_frame) == len(breathing_temperature_history)
assert set(breathing_temperature_frame.study_interval.unique()) == {2022, 2024}
breathing_temperature_variants = {
    "history_only": [],
    "history_temperature": ["temp_previous_vs_7d"],
    "history_breathing": breathing_temperature_columns,
    "history_breathing_temperature": breathing_temperature_columns + ["temp_previous_vs_7d"],
}
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def evaluate_breathing_temperature_models():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for label, features in breathing_temperature_variants.items():
        columns = history_feature_columns + features
        transforms = [("history", history_imputer, history_feature_columns)]
        if features:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True,
            ), features))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(breathing_temperature_frame)
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "test_pct_any_feature_missing": (
                    np.nan if not features else 100 * test[features].isna().any(axis=1).mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)


breathing_temperature_scores, breathing_temperature_parameters = evaluate_breathing_temperature_models()
assert len(breathing_temperature_scores) == 20 * len(breathing_temperature_variants)
assert breathing_temperature_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
breathing_temperature_mae_paired = breathing_temperature_scores.pivot(index="seed", columns="feature", values="MAE")
breathing_temperature_changes = breathing_temperature_mae_paired.subtract(breathing_temperature_mae_paired.history_only, axis=0)
breathing_temperature_summary = breathing_temperature_scores.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_any_feature_missing=("test_pct_any_feature_missing", "mean"),
)
breathing_temperature_summary["change_vs_history_days"] = breathing_temperature_changes.mean()
breathing_temperature_summary["better_than_history_splits"] = breathing_temperature_changes.lt(0).sum()
breathing_temperature_summary = breathing_temperature_summary.sort_values("mean_MAE_days")
with pd.option_context("display.max_columns", None):
    display(breathing_temperature_summary.round(3))
# Exploratory overlapping splits, not independent significance tests.
# Each wearable model also includes its missingness indicator. No files exported.


breathing_temperature_added_change = (
    breathing_temperature_mae_paired.history_breathing_temperature
    - breathing_temperature_mae_paired.history_breathing
)
print("Adding temperature to breathing: mean MAE change",
      round(breathing_temperature_added_change.mean(), 4), "days (negative is better).")
print("Temperature improves breathing-only in", int(breathing_temperature_added_change.lt(0).sum()), "of 20 splits.")
# Feature choices used earlier test results: exploratory, not an untouched final test.
# Keep outputs private; no exports or commits performed here.


In [ ]:
# History versus history + ONE latest daily signal; both intervals, 20 splits.
# "Latest" preserves prior experiments: exact lagged day, NOT latest nonmissing/forward-filled.
# Temperature, resting HR, breathing: D-1. HRV: night-start D-2 (includes morning D-1).
# TODO(INTERVAL2_REVIEW): HR uses the provisional daily minimum aggregation;
# repeated Interval 2 HR values and phase-derived labels remain unresolved.
latest_signal_cycles = baseline_cycles.loc[baseline_cycles.study_interval.isin([2022, 2024])].copy()
latest_signal_history = add_prior_cycle_history(
    data.make_daily_prediction_frame(latest_signal_cycles), latest_signal_cycles,
)
latest_signal_daily = wearables.load_daily_signals()
latest_signal_daily = latest_signal_daily.loc[latest_signal_daily.study_interval.isin([2022, 2024])].copy()

def build_latest_signal_features(daily, frame):
    out = frame.copy()
    for signal in wearables.SIGNALS:
        lagged = daily[wearables.KEY + [signal]].copy()
        lagged["day_in_study"] += 2 if signal == "hrv_rmssd" else 1
        lagged = lagged.rename(columns={signal: f"{signal}_latest"})
        out = out.merge(lagged, on=wearables.KEY, how="left", validate="one_to_one")
    return out

latest_signal_frame = build_latest_signal_features(latest_signal_daily, latest_signal_history)
assert len(latest_signal_frame) == len(latest_signal_history)
assert set(latest_signal_frame.study_interval.unique()) == {2022, 2024}
latest_signal_variants = {"history_only": []}
latest_signal_variants.update({f"history_{signal}": [f"{signal}_latest"] for signal in wearables.SIGNALS})
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def evaluate_latest_signal_models():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for label, features in latest_signal_variants.items():
        columns = history_feature_columns + features
        transforms = [("history", history_imputer, history_feature_columns)]
        if features:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True,
            ), features))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(latest_signal_frame)
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "test_pct_any_feature_missing": (
                    np.nan if not features else 100 * test[features].isna().any(axis=1).mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)


latest_signal_scores, latest_signal_parameters = evaluate_latest_signal_models()
assert len(latest_signal_scores) == 20 * len(latest_signal_variants)
assert latest_signal_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
latest_signal_mae_paired = latest_signal_scores.pivot(index="seed", columns="feature", values="MAE")
latest_signal_changes = latest_signal_mae_paired.subtract(latest_signal_mae_paired.history_only, axis=0)
latest_signal_summary = latest_signal_scores.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_any_feature_missing=("test_pct_any_feature_missing", "mean"),
)
latest_signal_summary["change_vs_history_days"] = latest_signal_changes.mean()
latest_signal_summary["better_than_history_splits"] = latest_signal_changes.lt(0).sum()
latest_signal_summary = latest_signal_summary.sort_values("mean_MAE_days")
with pd.option_context("display.max_columns", None):
    display(latest_signal_summary.round(3))
# Exploratory overlapping splits, not independent significance tests.
# Each wearable model also includes its missingness indicator. No files exported.


# Exploratory model comparisons; no output files exported. Clear outputs before committing.


In [ ]:
# History + planned combinations of latest daily signals; both intervals, 20 splits.
# Forest settings are tuned inside training; feature sets are compared exploratorily,
# NOT selected inside inner validation. No claim of a final unbiased winner.
# "Latest" preserves prior experiments: exact lagged day, NOT latest nonmissing/forward-filled.
# Temperature, resting HR, breathing: D-1. HRV: night-start D-2 (includes morning D-1).
# TODO(INTERVAL2_REVIEW): HR uses the provisional daily minimum aggregation;
# repeated Interval 2 HR values and phase-derived labels remain unresolved.
latest_combo_cycles = baseline_cycles.loc[baseline_cycles.study_interval.isin([2022, 2024])].copy()
latest_combo_history = add_prior_cycle_history(
    data.make_daily_prediction_frame(latest_combo_cycles), latest_combo_cycles,
)
latest_combo_daily = wearables.load_daily_signals()
latest_combo_daily = latest_combo_daily.loc[latest_combo_daily.study_interval.isin([2022, 2024])].copy()

def build_latest_combo_features(daily, frame):
    out = frame.copy()
    for signal in wearables.SIGNALS:
        lagged = daily[wearables.KEY + [signal]].copy()
        lagged["day_in_study"] += 2 if signal == "hrv_rmssd" else 1
        lagged = lagged.rename(columns={signal: f"{signal}_latest"})
        out = out.merge(lagged, on=wearables.KEY, how="left", validate="one_to_one")
    return out

latest_combo_frame = build_latest_combo_features(latest_combo_daily, latest_combo_history)
assert len(latest_combo_frame) == len(latest_combo_history)
assert set(latest_combo_frame.study_interval.unique()) == {2022, 2024}
latest_combo_variants = {
    "history_only": [],
    "history_breathing_temperature": ["breathing_rate_latest", "skin_temp_latest"],
    "history_breathing_hrv": ["breathing_rate_latest", "hrv_rmssd_latest"],
    "history_all_four": ["resting_hr_latest", "skin_temp_latest", "hrv_rmssd_latest", "breathing_rate_latest"],
}
# Raw daily summaries only: no rolling statistics or temperature-baseline deviations.
# All variants retain exactly the same prediction rows, including missing readings.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def evaluate_latest_combo_models():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for label, features in latest_combo_variants.items():
        columns = history_feature_columns + features
        transforms = [("history", history_imputer, history_feature_columns)]
        if features:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True,
            ), features))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(latest_combo_frame)
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "test_pct_any_feature_missing": (
                    np.nan if not features else 100 * test[features].isna().any(axis=1).mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)


latest_combo_scores, latest_combo_parameters = evaluate_latest_combo_models()
assert len(latest_combo_scores) == 20 * len(latest_combo_variants)
assert latest_combo_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
latest_combo_mae_paired = latest_combo_scores.pivot(index="seed", columns="feature", values="MAE")
latest_combo_changes = latest_combo_mae_paired.subtract(latest_combo_mae_paired.history_only, axis=0)
latest_combo_summary = latest_combo_scores.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_any_feature_missing=("test_pct_any_feature_missing", "mean"),
)
latest_combo_summary["change_vs_history_days"] = latest_combo_changes.mean()
latest_combo_summary["better_than_history_splits"] = latest_combo_changes.lt(0).sum()
latest_combo_summary = latest_combo_summary.sort_values("mean_MAE_days")
with pd.option_context("display.max_columns", None):
    display(latest_combo_summary.round(3))
# Exploratory overlapping splits, not independent significance tests.
# Each wearable model also includes its missingness indicator. No files exported.


# Exploratory model comparisons; no output files exported. Clear outputs before committing.


## Missingness Re-evaluation

The lack of effectiveness of wearables suggests there may be some missing data. Let's make sure:
1) Data isn't too sparse close to periods
2) Imputation (training data substituted for test data when unavailable) isn't being overly relied on.

We can also compare specific days where there are readings (no missingness) to see if the model improves from history only. 

### Coverage Checking First

In [ ]:
# History + planned combinations of latest daily signals; both intervals, 20 splits.
# Forest settings are tuned inside training; feature sets are compared exploratorily,
# NOT selected inside inner validation. No claim of a final unbiased winner.
# "Latest" preserves prior experiments: exact lagged day, NOT latest nonmissing/forward-filled.
# Temperature, resting HR, breathing: D-1. HRV: night-start D-2 (includes morning D-1).
# TODO(INTERVAL2_REVIEW): HR uses the provisional daily minimum aggregation;
# repeated Interval 2 HR values and phase-derived labels remain unresolved.
coverage_cycles = baseline_cycles.loc[baseline_cycles.study_interval.isin([2022, 2024])].copy()
coverage_history = add_prior_cycle_history(
    data.make_daily_prediction_frame(coverage_cycles), coverage_cycles,
)
coverage_daily = wearables.load_daily_signals()
coverage_daily = coverage_daily.loc[coverage_daily.study_interval.isin([2022, 2024])].copy()

def build_coverage_features(daily, frame):
    out = frame.copy()
    for signal in wearables.SIGNALS:
        lagged = daily[wearables.KEY + [signal]].copy()
        lagged["day_in_study"] += 2 if signal == "hrv_rmssd" else 1
        lagged = lagged.rename(columns={signal: f"{signal}_latest"})
        out = out.merge(lagged, on=wearables.KEY, how="left", validate="one_to_one")
    return out

coverage_frame = build_coverage_features(coverage_daily, coverage_history)
assert len(coverage_frame) == len(coverage_history)
assert set(coverage_frame.study_interval.unique()) == {2022, 2024}

# Diagnostic only: future period timing is used to group coverage, NEVER as a predictor.
# "Within W days" = target 1..W, inclusive, BEFORE the next recorded period.
# Windows overlap: 1..5 is contained in 1..7 and 1..9. Period day itself is excluded.
# Coverage concerns model-eligible cycle rows and lagged, cleaned daily values, not raw records.
from IPython.display import display
coverage_columns = [f"{s}_latest" for s in wearables.SIGNALS]
coverage_rows = []
for scope, frame in [("both", coverage_frame)] + [
    (str(interval), group) for interval, group in coverage_frame.groupby("study_interval")
]:
    for window in [5, 7, 9]:
        for label, mask in [("within", frame[TARGET].between(1, window)),
                            ("outside", frame[TARGET] > window)]:
            subset = frame.loc[mask]
            for feature in coverage_columns:
                missing = subset[feature].isna()
                coverage_rows.append(dict(
                    interval=scope, window_days=window, location=label, feature=feature,
                    rows=len(subset), participants=subset.id.nunique(),
                    missing_rows=int(missing.sum()), pct_missing=100 * missing.mean(),
                ))
coverage_near_period = pd.DataFrame(coverage_rows)
coverage_near_period_table = coverage_near_period.pivot(
    index=["interval", "feature", "window_days"], columns="location", values="pct_missing",
).rename(columns={"within": "within_pct_missing", "outside": "outside_pct_missing"})
coverage_near_period_table["within_minus_outside_pp"] = (
    coverage_near_period_table.within_pct_missing - coverage_near_period_table.outside_pct_missing
)
print("Missingness before the next period; positive difference means more missing nearby.")
with pd.option_context("display.max_columns", None):
    display(coverage_near_period_table.round(2))
print("Counts and denominators are retained in coverage_near_period. No data exported.")


In [ ]:
# Held-out imputation/error diagnostic: history versus history + all four latest values.
# Both intervals; 20 participant splits; forest tuning and imputation fit inside training.
# Requires the preceding coverage cell. Missing flags describe values BEFORE imputation.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend

def run_imputation_diagnostic():
    predictions, imputation_counts = [], []
    for seed in range(20):
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(coverage_frame)
        assert set(train.id).isdisjoint(test.id)
        folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
            train, train[TARGET], groups=train.id,
        ))
        heldout = test[wearables.KEY + [TARGET]].copy()
        heldout["seed"] = seed
        for feature in coverage_columns:
            heldout[feature + "_missing"] = test[feature].isna()
        for label, features in [("history", []), ("wearables", coverage_columns)]:
            transforms = [("history", SimpleImputer(strategy="median", keep_empty_features=True),
                           history_feature_columns)]
            if features:
                transforms.append(("wearable", SimpleImputer(
                    strategy="median", add_indicator=True, keep_empty_features=True,
                ), features))
            pipeline = Pipeline([
                ("prepare", ColumnTransformer(transforms, remainder="drop")),
                ("forest", RandomForestRegressor(n_estimators=100, max_features=1.0,
                                                  random_state=SEED, n_jobs=1)),
            ])
            search = GridSearchCV(pipeline,
                {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]},
                scoring="neg_mean_absolute_error", cv=folds, n_jobs=4, error_score="raise",
            )
            columns = history_feature_columns + features
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            prediction = search.predict(test[columns])
            assert np.isfinite(prediction).all()
            heldout[label + "_absolute_error"] = np.abs(test[TARGET].to_numpy() - prediction)
            if features:
                for name, cols in [("history", history_feature_columns), ("wearable", features)]:
                    imputer = search.best_estimator_.named_steps["prepare"].named_transformers_[name]
                    for feature, fill in zip(cols, imputer.statistics_):
                        imputation_counts.append(dict(seed=seed, feature=feature,
                            train_rows=len(train), train_imputed=int(train[feature].isna().sum()),
                            test_rows=len(test), test_imputed=int(test[feature].isna().sum()),
                            fill_value=fill, all_training_missing=train[feature].isna().all()))
        predictions.append(heldout)
        if (seed + 1) % 5 == 0:
            print(f"Imputation diagnostic: {seed + 1}/20 splits complete", flush=True)
    return pd.concat(predictions, ignore_index=True), pd.DataFrame(imputation_counts)

imputation_predictions, imputation_counts = run_imputation_diagnostic()
imputation_counts["train_pct_imputed"] = 100 * imputation_counts.train_imputed / imputation_counts.train_rows
imputation_counts["test_pct_imputed"] = 100 * imputation_counts.test_imputed / imputation_counts.test_rows
print("Mean percentage imputed across splits (including cycle-history predictors):")
display(imputation_counts.groupby("feature").agg(
    train_pct_imputed=("train_pct_imputed", "mean"), test_pct_imputed=("test_pct_imputed", "mean"),
    all_missing_training_splits=("all_training_missing", "sum"),
).round(2))
# Same day may be tested in several splits. Average its held-out absolute errors first,
# so repeatedly tested days do not receive extra weight in the following diagnostics.
missing_columns = [f + "_missing" for f in coverage_columns]
imputation_unique_days = imputation_predictions.groupby(wearables.KEY, as_index=False).agg({
    "history_absolute_error": "mean", "wearables_absolute_error": "mean",
    TARGET: "first", **{f: "first" for f in missing_columns},
})
imputation_unique_days["wearables_minus_history"] = (
    imputation_unique_days.wearables_absolute_error - imputation_unique_days.history_absolute_error
)
imputation_status_rows = []
for feature in coverage_columns:
    for missing, subset in imputation_unique_days.groupby(feature + "_missing"):
        imputation_status_rows.append(dict(feature=feature, status="imputed" if missing else "observed",
            unique_days=len(subset), participants=subset.id.nunique(),
            history_MAE=subset.history_absolute_error.mean(), wearable_MAE=subset.wearables_absolute_error.mean(),
            change_vs_history=subset.wearables_minus_history.mean()))
imputation_status_summary = pd.DataFrame(imputation_status_rows)
print("Matched-day errors: all-four wearable model vs history on the SAME rows. Negative change is better.")
with pd.option_context("display.max_columns", None):
    display(imputation_status_summary.round(3))
imputation_unique_days["n_wearables_imputed"] = imputation_unique_days[missing_columns].sum(axis=1)
print("Errors by number of wearable inputs imputed:")
display(imputation_unique_days.groupby("n_wearables_imputed").agg(
    unique_days=("id", "size"), history_MAE=("history_absolute_error", "mean"),
    wearable_MAE=("wearables_absolute_error", "mean"), change_vs_history=("wearables_minus_history", "mean"),
).round(3))
# One point per participant for descriptive Spearman correlations, not repeated daily samples.
imputation_by_participant = imputation_unique_days.groupby("id").agg({
    **{f: "mean" for f in missing_columns}, "wearables_absolute_error": "mean",
    "history_absolute_error": "mean", "wearables_minus_history": "mean",
})
imputation_correlations = pd.DataFrame([
    dict(feature=f, participants=len(imputation_by_participant),
         spearman_missing_vs_wearable_MAE=imputation_by_participant[f + "_missing"].corr(
             imputation_by_participant.wearables_absolute_error, method="spearman"),
         spearman_missing_vs_MAE_change=imputation_by_participant[f + "_missing"].corr(
             imputation_by_participant.wearables_minus_history, method="spearman"))
    for f in coverage_columns
])
print("Participant-level associations: positive means more missingness accompanies higher error / worse wearable gain.")
with pd.option_context("display.max_columns", None):
    display(imputation_correlations.round(3))
print("Descriptive only, not causal or significance tests: interval, cycle timing and participant difficulty can confound associations.")
# Flags can overlap across features. All-empty training features use zero fallback, not a median.
# No files exported; clear notebook outputs before committing.


In [ ]:
# History + planned combinations of latest daily signals; Interval 2 ONLY, 20 splits.
# Both models are random forests: history only vs history + all four latest signals.
# No Interval 1 rows used for fitting or earlier-cycle history.
# Retain original participant assignments before restricting to Interval 2.
# Forest settings are tuned inside training; feature sets are compared exploratorily,
# NOT selected inside inner validation. No claim of a final unbiased winner.
# "Latest" preserves prior experiments: exact lagged day, NOT latest nonmissing/forward-filled.
# Temperature, resting HR, breathing: D-1. HRV: night-start D-2 (includes morning D-1).
# TODO(INTERVAL2_REVIEW): HR uses the provisional daily minimum aggregation;
# repeated Interval 2 HR values and phase-derived labels remain unresolved.
interval2_latest_cycles = baseline_cycles.loc[baseline_cycles.study_interval.isin([2024])].copy()
interval2_latest_history = add_prior_cycle_history(
    data.make_daily_prediction_frame(interval2_latest_cycles), interval2_latest_cycles,
)
interval2_latest_daily = wearables.load_daily_signals()
interval2_latest_daily = interval2_latest_daily.loc[interval2_latest_daily.study_interval.isin([2024])].copy()

def build_interval2_latest_features(daily, frame):
    out = frame.copy()
    for signal in wearables.SIGNALS:
        lagged = daily[wearables.KEY + [signal]].copy()
        lagged["day_in_study"] += 2 if signal == "hrv_rmssd" else 1
        lagged = lagged.rename(columns={signal: f"{signal}_latest"})
        out = out.merge(lagged, on=wearables.KEY, how="left", validate="one_to_one")
    return out

interval2_latest_frame = build_interval2_latest_features(interval2_latest_daily, interval2_latest_history)
assert len(interval2_latest_frame) == len(interval2_latest_history)
assert set(interval2_latest_frame.study_interval.unique()) == {2024}
interval2_latest_variants = {
    "history_only": [],
    "history_all_four": ["resting_hr_latest", "skin_temp_latest", "hrv_rmssd_latest", "breathing_rate_latest"],
}
# Raw daily summaries only: no rolling statistics or temperature-baseline deviations.
# All variants retain exactly the same prediction rows, including missing readings.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def evaluate_interval2_latest_models():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for label, features in interval2_latest_variants.items():
        columns = history_feature_columns + features
        transforms = [("history", history_imputer, history_feature_columns)]
        if features:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=True, keep_empty_features=True,
            ), features))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(interval2_latest_frame)
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "n_train_participants": train.id.nunique(),
                "n_test_participants": test.id.nunique(),
                "test_pct_any_feature_missing": (
                    np.nan if not features else 100 * test[features].isna().any(axis=1).mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)


interval2_latest_scores, interval2_latest_parameters = evaluate_interval2_latest_models()
assert len(interval2_latest_scores) == 20 * len(interval2_latest_variants)
assert interval2_latest_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
interval2_latest_mae_paired = interval2_latest_scores.pivot(index="seed", columns="feature", values="MAE")
interval2_latest_changes = interval2_latest_mae_paired.subtract(interval2_latest_mae_paired.history_only, axis=0)
interval2_latest_summary = interval2_latest_scores.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_any_feature_missing=("test_pct_any_feature_missing", "mean"),
)
interval2_latest_summary["change_vs_history_days"] = interval2_latest_changes.mean()
interval2_latest_summary["better_than_history_splits"] = interval2_latest_changes.lt(0).sum()
interval2_latest_summary = interval2_latest_summary.sort_values("mean_MAE_days")
with pd.option_context("display.max_columns", None):
    display(interval2_latest_summary.round(3))
# Exploratory overlapping splits, not independent significance tests.
# Each wearable model also includes its missingness indicator. No files exported.


# Exploratory model comparisons; no output files exported. Clear outputs before committing.

print("Interval 2 cohort:", interval2_latest_frame.id.nunique(), "participants;",
      len(interval2_latest_frame), "prediction rows.")
print("Train/test participant counts across splits:")
display(interval2_latest_scores.loc[interval2_latest_scores.feature.eq("history_only"),
    ["n_train_participants", "n_test_participants"]].agg(["min", "max"]))
print("Coverage (before imputation):")
display((100 * interval2_latest_frame[[f"{s}_latest" for s in wearables.SIGNALS]].isna().mean()).rename("pct_missing").round(2))
# This changes the training cohort and test mix, not only coverage. Not a causal test of missingness.


### Try new Imputation Methods

Instead of accepting training median to replace, let's test 1- participants recent (past) median and 2- carry-forward past readings

In [ ]:
# Interval 2: compare filling methods using identical rows, flags and reading ages.
# Exploratory fixed choices, not chosen from test results. TODO(INTERVAL2_REVIEW).
# Age = extra calendar days since the most recent reading eligible under our lag rule.
# HRV lag is 2 night-label days; other signals lag 1 day. Never use today's/future readings.
PERSONAL_MEDIAN_DAYS = 7
PERSONAL_MIN_READINGS = 2
MAX_CARRY_EXTRA_DAYS = 2
fill_cycles = baseline_cycles.loc[baseline_cycles.study_interval.eq(2024)].copy()
fill_history = add_prior_cycle_history(data.make_daily_prediction_frame(fill_cycles), fill_cycles)
fill_daily = wearables.load_daily_signals()
fill_daily = fill_daily.loc[fill_daily.study_interval.eq(2024)].copy()
fill_values = [f"{s}_latest" for s in wearables.SIGNALS]
fill_context = [f"{s}_{kind}" for s in wearables.SIGNALS for kind in ("original_missing", "age_extra_days")]

def build_past_only_fill_frames(daily, frame):
    keys = pd.concat([daily[wearables.KEY], frame[wearables.KEY]]).drop_duplicates()
    dense = wearables.dense_days(keys).merge(daily, on=wearables.KEY, how="left", validate="one_to_one")
    parts = {method: [] for method in ("population_median", "personal_median", "limited_carry")}
    for _, person in dense.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        outputs = {method: person[wearables.KEY].copy() for method in parts}
        for signal in wearables.SIGNALS:
            eligible = person[signal].shift(2 if signal == "hrv_rmssd" else 1)
            last_day = person.day_in_study.where(eligible.notna()).ffill()
            age = person.day_in_study - last_day
            # Recent median uses the 7 eligible calendar slots ending at the lagged day.
            # Missing current slot means the estimate uses only earlier actual readings.
            recent = eligible.rolling(PERSONAL_MEDIAN_DAYS, min_periods=PERSONAL_MIN_READINGS).median()
            carried = eligible.ffill().where(age <= MAX_CARRY_EXTRA_DAYS)
            values = {"population_median": eligible,
                      "personal_median": eligible.fillna(recent),
                      "limited_carry": eligible.fillna(carried)}
            for method, out in outputs.items():
                out[f"{signal}_latest"] = values[method]
                out[f"{signal}_original_missing"] = eligible.isna().astype(int)
                out[f"{signal}_age_extra_days"] = age.fillna(-1)  # -1 = no earlier eligible reading.
        for method in parts:
            parts[method].append(outputs[method])
    return {method: frame.merge(pd.concat(pieces, ignore_index=True), on=wearables.KEY,
                               how="left", validate="one_to_one") for method, pieces in parts.items()}

fill_frames = build_past_only_fill_frames(fill_daily, fill_history)
for method, frame in fill_frames.items():
    assert len(frame) == len(fill_history)
    assert frame.study_interval.eq(2024).all()
    assert frame[fill_context].notna().all().all()
    assert frame[fill_context].equals(fill_frames["population_median"][fill_context])
    for value in fill_values:
        observed = fill_frames["population_median"][value].notna()
        assert frame.loc[observed, value].equals(fill_frames["population_median"].loc[observed, value])
print("Fraction of rows still needing population-median fallback:")
display(pd.DataFrame({method: 100 * frame[fill_values].isna().mean()
                      for method, frame in fill_frames.items()}).round(2))
fill_variants = {"history_only": [], **{method: fill_values + fill_context for method in fill_frames}}
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def evaluate_fill_models():
    # Match the original history-only preprocessing (no redundant history indicator).
    history_imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]}
    rows, parameters = [], []
    for label, features in fill_variants.items():
        columns = history_feature_columns + features
        transforms = [("history", history_imputer, history_feature_columns)]
        if features:
            transforms.append(("wearable", SimpleImputer(
                strategy="median", add_indicator=False, keep_empty_features=True,
            ), features))
        pipeline = Pipeline([
            ("prepare", ColumnTransformer(transforms, remainder="drop")),
            ("forest", RandomForestRegressor(
                n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1,
            )),
        ])
        for seed in range(20):
            split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
            train, test = split.apply(fill_frames["population_median" if label == "history_only" else label])
            assert set(train.id).isdisjoint(test.id)
            if train.id.nunique() < 2 or test.empty:
                raise ValueError("Insufficient participants for nested evaluation.")
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(
                train, train[TARGET], groups=train.id,
            ))
            search = GridSearchCV(
                pipeline, param_grid=grid, scoring="neg_mean_absolute_error",
                cv=folds, refit=True, error_score="raise", n_jobs=4,
            )
            # Parallelize candidate fits, not trees as well (avoid nested parallelism).
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            predicted = search.predict(test[columns])
            assert np.isfinite(predicted).all()
            error = np.abs(test[TARGET].to_numpy() - predicted)
            rows.append({
                "feature": label, "seed": seed, "MAE": error.mean(),
                "participant_mean_MAE": pd.Series(
                    error, index=test.id.to_numpy()
                ).groupby(level=0).mean().mean(),
                "n_test_rows": len(test),
                "n_train_participants": train.id.nunique(),
                "n_test_participants": test.id.nunique(),
                "test_pct_any_feature_missing": (
                    np.nan if not features else 100 * test[fill_values].isna().any(axis=1).mean()
                ),
            })
            parameters.append({"feature": label, "seed": seed, **search.best_params_})
        completed = [r["MAE"] for r in rows if r["feature"] == label]
        print(f"{label}: 20 splits complete, mean MAE {np.mean(completed):.4f}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(parameters)


fill_scores, fill_parameters = evaluate_fill_models()
assert len(fill_scores) == 20 * len(fill_variants)
assert fill_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
fill_mae_paired = fill_scores.pivot(index="seed", columns="feature", values="MAE")
fill_changes = fill_mae_paired.subtract(fill_mae_paired.history_only, axis=0)
fill_summary = fill_scores.groupby("feature").agg(
    mean_MAE_days=("MAE", "mean"), split_SD_days=("MAE", "std"),
    mean_participant_MAE_days=("participant_mean_MAE", "mean"),
    mean_test_pct_any_feature_missing=("test_pct_any_feature_missing", "mean"),
)
fill_summary["change_vs_history_days"] = fill_changes.mean()
fill_summary["better_than_history_splits"] = fill_changes.lt(0).sum()
fill_summary = fill_summary.sort_values("mean_MAE_days")
with pd.option_context("display.max_columns", None):
    display(fill_summary.round(3))
# Exploratory overlapping splits, not independent significance tests.
# Original missingness flags and reading ages are identical across all three filling methods.


# Exploratory model comparisons; no output files exported. Clear outputs before committing.


fill_method_changes = fill_mae_paired.subtract(fill_mae_paired.population_median, axis=0)
print("Filling methods vs the matched population-median control (negative change is better):")
display(pd.DataFrame({"mean_MAE_change_days": fill_method_changes.mean(),
                      "better_splits": fill_method_changes.lt(0).sum()}).loc[list(fill_frames)].round(4))
print("This population-median control includes reading-age features, unlike earlier experiments.")
print("Past-only personal statistics may use earlier readings from test participants, never future readings or target values.")
# Median fallback and model tuning are fitted within training folds. No interpolation/backfill.
# No outputs exported. Clear notebook outputs before committing.


In [ ]:
# Interval 2: held-out errors by ORIGINAL availability, before any filling.
# Compare history-only against history + all four latest signals, limited carry-forward.
# Requires fill_frames / fill_values / fill_context from the preceding imputation cell.
# TODO(INTERVAL2_REVIEW): exploratory; existing data-quality caveats still apply.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend

availability_frame = fill_frames["limited_carry"].copy()
assert availability_frame.study_interval.eq(2024).all()
availability_flags = [f"{s}_original_missing" for s in wearables.SIGNALS]

def evaluate_by_availability():
    results = []
    for seed in range(20):
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(availability_frame)
        assert set(train.id).isdisjoint(test.id)
        folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(train, train[TARGET], groups=train.id))
        out = test[wearables.KEY + [TARGET] + availability_flags].copy()
        out["seed"] = seed
        for label, features in [("history", []), ("wearables", fill_values + fill_context)]:
            transforms = [("history", SimpleImputer(strategy="median", keep_empty_features=True), history_feature_columns)]
            if features:
                transforms.append(("wearables", SimpleImputer(strategy="median", keep_empty_features=True), features))
            model = Pipeline([
                ("prepare", ColumnTransformer(transforms)),
                ("forest", RandomForestRegressor(n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1)),
            ])
            search = GridSearchCV(model,
                {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]},
                scoring="neg_mean_absolute_error", cv=folds, n_jobs=4, error_score="raise")
            columns = history_feature_columns + features
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            pred = search.predict(test[columns])
            assert np.isfinite(pred).all()
            out[label + "_error"] = np.abs(test[TARGET].to_numpy() - pred)
        results.append(out)
        if (seed + 1) % 5 == 0:
            print(f"Availability comparison: {seed + 1}/20 splits complete", flush=True)
    return pd.concat(results, ignore_index=True)

availability_predictions = evaluate_by_availability()
# Average absolute errors for each unique day across its held-out appearances.
# Do not average predictions first; do not count repeatedly tested days as independent.
availability_days = availability_predictions.groupby(wearables.KEY, as_index=False).agg({
    TARGET: "first", "history_error": "mean", "wearables_error": "mean",
    **{f: "first" for f in availability_flags},
})
availability_days["any_missing"] = availability_days[availability_flags].astype(bool).any(axis=1)
availability_rows = []
for scope, flag in [("all_four", "any_missing")] + list(zip(wearables.SIGNALS, availability_flags)):
    for missing in [False, True]:
        group = availability_days.loc[availability_days[flag].astype(bool).eq(missing)]
        availability_rows.append(dict(
            feature=scope, availability="missing" if missing else "observed",
            unique_days=len(group), participants=group.id.nunique(),
            history_MAE=group.history_error.mean(), wearable_MAE=group.wearables_error.mean(),
            change_vs_history=(group.wearables_error - group.history_error).mean(),
        ))
availability_summary = pd.DataFrame(availability_rows)
print("Original readings: all_four missing = at least one unavailable; observed = all four available.")
print("Both models scored on identical rows in every group. Negative change favors wearables.")
with pd.option_context("display.max_columns", None):
    display(availability_summary.round(3))
# Also retain the original mean-of-split MAE convention for comparability with earlier cells.
availability_predictions["any_missing"] = availability_predictions[availability_flags].astype(bool).any(axis=1)
availability_per_split = availability_predictions.groupby(["seed", "any_missing"]).agg(
    rows=("id", "size"), history_MAE=("history_error", "mean"), wearable_MAE=("wearables_error", "mean"),
)
availability_per_split["change_vs_history"] = availability_per_split.wearable_MAE - availability_per_split.history_MAE
print("Mean across splits, by availability; absent groups are not treated as zero error:")
display(availability_per_split.groupby("any_missing").agg(
    splits=("rows", "size"), history_MAE=("history_MAE", "mean"),
    wearable_MAE=("wearable_MAE", "mean"), change_vs_history=("change_vs_history", "mean"),
).round(3))
print("Overall mean-of-split MAE:")
display(availability_predictions.groupby("seed")[["history_error", "wearables_error"]].mean().mean().round(4))
print("Descriptive associations only: availability groups differ in participants and cycle timing. No causal or significance claim.")
# All feature-specific groups use the SAME all-four wearable model, not single-feature models.
# Original missing flags remain set even if carry-forward supplies a value. No exports.


### Let's see where the history model does well and poorly-- to glean information about on if and where the wearables models might improve predictions

In [ ]:
# Interval 2: rank held-out participant MAE, worst history performance first.
# Wearable comparator = history + all four latest signals, limited carry-forward.
# History alone remains best overall; this is the best wearable variant in our filling comparison.
# Requires the performance-by-availability cell; reuses predictions, does not refit models.
if "availability_predictions" not in globals() or "fill_cycles" not in globals():
    raise RuntimeError("Run the Interval 2 imputation and performance-by-availability cells first.")
assert availability_predictions.study_interval.eq(2024).all()
assert fill_cycles.study_interval.eq(2024).all()
# Average each day's absolute errors over its held-out appearances, then average days per person.
ranking_days = availability_predictions.groupby(wearables.KEY, as_index=False).agg(
    history_error=("history_error", "mean"), wearable_error=("wearables_error", "mean"),
)
assert ranking_days[["history_error", "wearable_error"]].notna().all().all()
participant_error_ranking = ranking_days.groupby("id").agg(
    history_MAE_days=("history_error", "mean"), wearable_MAE_days=("wearable_error", "mean"),
    evaluated_days=("day_in_study", "size"),
)
# Cycle summaries are one value per completed eligible cycle, NOT weighted by daily rows.
# Counts refer to Interval 2 cycles passing the existing 15–60-day filters; not all study cycles.
participant_cycle_summary = fill_cycles.groupby("id").agg(
    mean_cycle_length_days=("cycle_length", "mean"),
    eligible_cycles=("cycle_length", "size"),
    min_cycle_length_days=("cycle_length", "min"),
    max_cycle_length_days=("cycle_length", "max"),
)
participant_error_ranking = participant_error_ranking.join(participant_cycle_summary, validate="one_to_one")
assert participant_error_ranking.eligible_cycles.notna().all()
participant_error_ranking["wearable_minus_history_days"] = (
    participant_error_ranking.wearable_MAE_days - participant_error_ranking.history_MAE_days
)
participant_error_ranking = participant_error_ranking.sort_values("history_MAE_days", ascending=False)
participant_error_ranking.insert(0, "rank_worst_first", np.arange(1, len(participant_error_ranking) + 1))
participant_error_ranking = participant_error_ranking[[
    "rank_worst_first", "history_MAE_days", "wearable_MAE_days", "wearable_minus_history_days",
    "mean_cycle_length_days", "eligible_cycles", "min_cycle_length_days", "max_cycle_length_days", "evaluated_days",
]]
print("Interval 2 — worst to best history MAE; negative wearable-minus-history means wearables help.")
print("Cycle statistics include completed eligible cycles only (15–60 days); few cycles limit interpretation.")
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(participant_error_ranking.round(3))
# Descriptive retrospective diagnostics, not extra model inputs. No files exported.


Finding 1: Worse history MAE on cycles that are more than 3 days off from the population median. We should see if the worse MAE occurs when cycles within a participant are irregular or an individual is consistently above or below population median.

Finding 2: Some partipants have the same median cycles, and very close history only MAE, but very different wearables scores. We can look into missingness between the two, whether predictions were too early or late, and how prediction errors track across different days of cycle.

In [ ]:
# Original feature availability on model prediction days, BEFORE carry-forward/imputation.
# Interval 2; labels describe this exploratory comparison, not stable participant traits.
participant_focus = {41: "Wearable-strong participant", 20: "History-strong participant"}
if "fill_frames" not in globals():
    raise RuntimeError("Run the past-only imputation setup cell first.")
focus_frame = fill_frames["limited_carry"]
focus_missing_fig, focus_missing_axes = plt.subplots(4, 2, figsize=(14, 9), sharex="col", sharey=True, layout="constrained")
focus_missing_counts = []
for col, (participant, label) in enumerate(participant_focus.items()):
    person = focus_frame.loc[focus_frame.id.eq(participant) & focus_frame.study_interval.eq(2024)].sort_values("day_in_study")
    cycles = fill_cycles.loc[fill_cycles.id.eq(participant)].sort_values("start_day")
    if person.empty or cycles.empty:
        raise ValueError(f"No Interval 2 modelling rows for participant {participant}.")
    origin, end = int(cycles.start_day.min()), int(cycles.next_start_day.max())
    days = np.arange(origin, end)
    person = person.set_index("day_in_study").reindex(days)
    boundaries = np.unique(np.r_[cycles.start_day, cycles.next_start_day])
    for row, signal in enumerate(wearables.SIGNALS):
        ax = focus_missing_axes[row, col]
        missing = person[f"{signal}_original_missing"]
        ax.step(days - origin, missing, where="mid", color="C0", linewidth=1.2)
        ax.scatter((days - origin)[missing.eq(1)], missing.loc[missing.eq(1)], color="C1", marker="x", s=22)
        for day in boundaries:
            ax.axvline(day - origin, color="red", linestyle=":", linewidth=1, alpha=0.65)
        ax.set_yticks([0, 1], ["Observed", "Missing"])
        ax.set_ylim(-0.2, 1.25)
        ax.set_xlim(-0.5, end - origin + 0.5)
        ax.set_title(signal.replace("_", " ") + (f" — {label} (ID {participant})" if row == 0 else ""), fontsize=10)
        if row == 3:
            ax.set_xlabel("Days since first evaluated period (red lines = period starts)")
        focus_missing_counts.append(dict(id=participant, feature=signal, missing_days=int(missing.sum()), evaluated_days=int(missing.notna().sum())))
focus_missing_fig.suptitle("Original missingness — prior-day model inputs, not filled values", fontsize=13)
plt.show()
display(pd.DataFrame(focus_missing_counts))
# HRV uses D-2 night-start labels; other signals D-1. Gaps outside eligible cycles stay blank.
# No participant outputs exported; clear outputs before committing.


In [ ]:
# Refit identical models only for splits where ID 41 or 20 is held out.
# Train on all original Interval 2 training participants, NOT just these two people.
# Wearable model = history + all four latest values, limited carry + flags and ages.
# Refit needed because previous diagnostics retained absolute errors, not predictions.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend

def evaluate_focus_predictions():
    results = []
    for seed in range(20):
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(focus_frame)
        test = test.loc[test.id.isin(participant_focus)].copy()
        if test.empty:
            continue
        assert set(train.id).isdisjoint(test.id)
        folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(train, train[TARGET], groups=train.id))
        out = test[wearables.KEY + [TARGET, "cycle_day", "cycle_index"]].copy()
        out["seed"] = seed
        for label, features in [("history", []), ("wearables", fill_values + fill_context)]:
            transforms = [("history", SimpleImputer(strategy="median", keep_empty_features=True), history_feature_columns)]
            if features:
                transforms.append(("wearables", SimpleImputer(strategy="median", keep_empty_features=True), features))
            model = Pipeline([
                ("prepare", ColumnTransformer(transforms)),
                ("forest", RandomForestRegressor(n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1)),
            ])
            search = GridSearchCV(model,
                {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]},
                scoring="neg_mean_absolute_error", cv=folds, n_jobs=4, error_score="raise")
            columns = history_feature_columns + features
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            pred = search.predict(test[columns])
            assert np.isfinite(pred).all()
            out[label + "_prediction"] = pred
            out[label + "_signed_error"] = pred - test[TARGET].to_numpy()
            out[label + "_error"] = np.abs(test[TARGET].to_numpy() - pred)
        results.append(out)
        if (seed + 1) % 5 == 0:
            print(f"Participant prediction plots: {seed + 1}/20 splits complete", flush=True)
    return pd.concat(results, ignore_index=True)


focus_predictions = evaluate_focus_predictions()
focus_daily_predictions = focus_predictions.groupby(wearables.KEY, as_index=False).agg({
    TARGET: "first", "cycle_day": "first", "cycle_index": "first",
    "history_prediction": "mean", "wearables_prediction": "mean",
    "history_error": "mean", "wearables_error": "mean",
    "history_signed_error": "mean", "wearables_signed_error": "mean", "seed": "nunique",
}).rename(columns={"seed": "heldout_splits"})
focus_prediction_figures = {}
for participant, label in participant_focus.items():
    person = focus_daily_predictions.loc[focus_daily_predictions.id.eq(participant)].sort_values("day_in_study")
    cycles = fill_cycles.loc[fill_cycles.id.eq(participant)].sort_values("start_day")
    origin, end = int(cycles.start_day.min()), int(cycles.next_start_day.max())
    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, layout="constrained")
    # Plot cycles separately to avoid a misleading connecting line across period resets.
    for i, cycle in enumerate(cycles.itertuples()):
        current = person.loc[person.cycle_index.eq(cycle.cycle_index)].set_index("day_in_study")
        current = current.reindex(np.arange(cycle.start_day, cycle.next_start_day))
        x = current.index.to_numpy() - origin
        axes[0].plot(x, current[TARGET], color="0.2", linestyle=":", linewidth=2, label="Actual days remaining" if i == 0 else None)
        for model, name, color, style in [("history", "History only", "C0", "-"), ("wearables", "History + wearables", "C1", "--")]:
            axes[0].plot(x, current[model + "_prediction"], color=color, linestyle=style, label=name if i == 0 else None)
            axes[1].plot(x, current[model + "_error"], color=color, linestyle=style, label=name if i == 0 else None)
        axes[0].text((cycle.start_day + cycle.next_start_day) / 2 - origin, 1.01,
                     f"Cycle {i + 1}: {cycle.cycle_length} days", transform=axes[0].get_xaxis_transform(),
                     ha="center", fontsize=9)
    for ax in axes:
        for i, day in enumerate(np.unique(np.r_[cycles.start_day, cycles.next_start_day])):
            ax.axvline(day - origin, color="red", linestyle=":", alpha=0.7, label="Recorded period start" if i == 0 else None)
        ax.set_xlim(-0.5, end - origin + 0.5)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.2)
    axes[0].set_ylabel("Predicted / actual days until next period")
    axes[0].legend(ncol=2, fontsize=9)
    axes[1].set_ylabel("Daily MAE (days)")
    axes[1].set_xlabel("Days since first evaluated period (continuous across cycles)")
    fig.suptitle(f"{label} (ID {participant}) — Interval 2\nAbove actual line = predicted too late; below = too early", fontsize=12)
    focus_prediction_figures[participant] = fig
    plt.show()
print("Lines average held-out predictions across splits; daily MAE averages absolute errors, not the error of the averaged prediction.")
print("Signed errors are retained in focus_daily_predictions: negative = early, positive = late.")
# Descriptive, selected-participant plots only; no statistical-significance claim. No exports.


Notice that as period days get closer, we start to plateau in average days until cycle. This is *likely* because our training data predicts day 1 of cycle to day 28 for example, and then the next day is day 1 again, so the random forest model predicts the average of days remaining, which is a minimum of 1. Let's investigate all participants a few days out from the period and see MAE

In [ ]:
# Interval 2: predictions when actual remaining time is exactly 1, 2 or 3 days.
# History RF vs history + latest wearables RF, with limited carry-forward.
# Refit identical 20 splits because earlier all-participant diagnostics saved only absolute errors.
# Training/inner tuning still uses ALL eligible training days, not just near-period days.
if "fill_frames" not in globals():
    raise RuntimeError("Run the past-only imputation comparison setup first.")
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend

near_period_frame = fill_frames["limited_carry"].copy()
assert near_period_frame.study_interval.eq(2024).all()
near_period_flags = [f"{s}_original_missing" for s in wearables.SIGNALS]

def evaluate_near_period_predictions():
    results = []
    for seed in range(20):
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(near_period_frame)
        assert set(train.id).isdisjoint(test.id)
        folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(train, train[TARGET], groups=train.id))
        out = test[wearables.KEY + [TARGET] + near_period_flags].copy()
        out["seed"] = seed
        for label, features in [("history", []), ("wearables", fill_values + fill_context)]:
            transforms = [("history", SimpleImputer(strategy="median", keep_empty_features=True), history_feature_columns)]
            if features:
                transforms.append(("wearables", SimpleImputer(strategy="median", keep_empty_features=True), features))
            model = Pipeline([
                ("prepare", ColumnTransformer(transforms)),
                ("forest", RandomForestRegressor(n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1)),
            ])
            search = GridSearchCV(model,
                {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [10, 30]},
                scoring="neg_mean_absolute_error", cv=folds, n_jobs=4, error_score="raise")
            columns = history_feature_columns + features
            with parallel_backend("threading", n_jobs=4):
                search.fit(train[columns], train[TARGET])
            pred = search.predict(test[columns])
            assert np.isfinite(pred).all()
            out[label + "_prediction"] = pred
            out[label + "_within_one"] = np.abs(test[TARGET].to_numpy() - pred) <= 1
            out[label + "_error"] = np.abs(test[TARGET].to_numpy() - pred)
        results.append(out.loc[out[TARGET].isin([1, 2, 3])].copy())
        if (seed + 1) % 5 == 0:
            print(f"Near-period predictions: {seed + 1}/20 splits complete", flush=True)
    return pd.concat(results, ignore_index=True)


near_period_predictions = evaluate_near_period_predictions()
assert not near_period_predictions.duplicated(wearables.KEY + ["seed"]).any()
assert near_period_predictions.groupby(wearables.KEY)[TARGET].nunique().eq(1).all()
# One point per unique participant-day: mean prediction across its held-out appearances.
# Retain mean absolute errors/success rates BEFORE averaging predictions to avoid cancellation.
near_period_days = near_period_predictions.groupby(wearables.KEY, as_index=False).agg({
    TARGET: "first", "seed": "nunique",
    **{f"{m}_{metric}": "mean" for m in ["history", "wearables"]
       for metric in ["prediction", "error", "within_one"]},
}).rename(columns={"seed": "heldout_splits"})
near_period_rows = []
near_period_fig, near_period_axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True, layout="constrained")
for ax, truth in zip(near_period_axes, [1, 2, 3]):
    group = near_period_days.loc[near_period_days[TARGET].eq(truth)]
    if group.empty:
        ax.set_title(f"Truth = {truth} days: no rows")
        continue
    samples = [group[f"{model}_prediction"].to_numpy() for model in ["history", "wearables"]]
    boxes = ax.boxplot(samples, positions=[1, 2], widths=0.45, patch_artist=True, showfliers=False)
    for box, color in zip(boxes["boxes"], ["C0", "C1"]):
        box.set_facecolor(color)
        box.set_alpha(0.25)
    for pos, (model, values, color) in enumerate(zip(["history", "wearables"], samples, ["C0", "C1"]), start=1):
        jitter = np.random.default_rng(SEED).uniform(-0.12, 0.12, len(values))
        ax.scatter(pos + jitter, values, color=color, alpha=0.65, s=18)
        near_period_rows.append(dict(
            actual_days=truth, model=model, unique_days=len(group), participants=group.id.nunique(),
            min_prediction=values.min(), median_prediction=np.median(values), max_prediction=values.max(),
            heldout_MAE=group[f"{model}_error"].mean(),
            pct_within_one_day=100 * group[f"{model}_within_one"].mean(),
        ))
    ax.axhline(truth, color="red", linestyle="--", linewidth=1.5, label="Actual remaining days")
    ax.set_xticks([1, 2], ["History", "History + wearables"])
    ax.set_title(f"Truth = {truth} day(s) | {len(group)} unique days")
    ax.grid(axis="x", visible=False)
    ax.legend(fontsize=8, loc="upper right")
near_period_axes[0].set_ylabel("Predicted days remaining (held-out mean per day)")
near_period_axes[0].set_ylim(bottom=0)
near_period_fig.suptitle("Do predictions approach 1–3 days when the period is imminent?")
plt.show()
near_period_summary = pd.DataFrame(near_period_rows)
with pd.option_context("display.max_columns", None):
    display(near_period_summary.round(3))
print("Each dot is one participant-day, averaged across held-out splits. Min/median/max describe these dots.")
print("MAE and ±1-day success are calculated on individual held-out predictions, then averaged per unique day.")
print("Cycles within a participant and repeated splits are correlated: these are descriptive distributions, not significance tests.")
print("The target starts at 1, so random-forest averaging cannot predict below 1. No hard multi-day floor is imposed.")
# TODO(INTERVAL2_REVIEW): existing data-quality caveats apply. No files exported.


### Finding: effective floor at 3 days. Final group allows 10 or 30 training rows and then averages the results, let's narrow a leaf to contain 1, 3, and 5 training rows. Test training MAE vs held out group to make sure this doesn't result in overfitting.

In [ ]:
# Interval 2: isolate minimum leaf size, retaining the existing depth grid [3, 6].
# Train-only depth tuning for EACH size; outer results compare sizes exploratorily.
# Uses history + all four latest signals, limited carry-forward + flags + ages.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend

if "fill_frames" not in globals():
    raise RuntimeError("Run the past-only imputation comparison setup first.")
leaf_frame = fill_frames["limited_carry"].copy()
assert leaf_frame.study_interval.eq(2024).all()
LEAF_SIZES = [1, 3, 5, 10, 30]

def compare_leaf_sizes():
    scores, predictions = [], []
    for label, features in [("history", []), ("history_wearables", fill_values + fill_context)]:
        columns = history_feature_columns + features
        for leaf_size in LEAF_SIZES:
            for seed in range(20):
                split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
                train, test = split.apply(leaf_frame)
                assert set(train.id).isdisjoint(test.id)
                folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(train, train[TARGET], groups=train.id))
                transforms = [("history", SimpleImputer(strategy="median", keep_empty_features=True), history_feature_columns)]
                if features:
                    transforms.append(("wearables", SimpleImputer(strategy="median", keep_empty_features=True), features))
                pipeline = Pipeline([
                    ("prepare", ColumnTransformer(transforms)),
                    ("forest", RandomForestRegressor(n_estimators=100, min_samples_leaf=leaf_size,
                        max_features=1.0, random_state=SEED, n_jobs=1)),
                ])
                search = GridSearchCV(pipeline, {"forest__max_depth": [3, 6]},
                    scoring="neg_mean_absolute_error", cv=folds, n_jobs=4, error_score="raise")
                with parallel_backend("threading", n_jobs=4):
                    search.fit(train[columns], train[TARGET])
                train_error = np.abs(search.predict(train[columns]) - train[TARGET].to_numpy())
                pred = search.predict(test[columns])
                signed = pred - test[TARGET].to_numpy()
                error = np.abs(signed)
                near = test[TARGET].between(1, 3).to_numpy()
                scores.append(dict(model=label, leaf_size=leaf_size, seed=seed,
                    selected_depth=search.best_params_["forest__max_depth"],
                    inner_CV_MAE=-search.best_score_, train_MAE=train_error.mean(),
                    test_MAE=error.mean(), test_participant_MAE=pd.Series(error, index=test.id).groupby(level=0).mean().mean(),
                    final_3d_MAE=error[near].mean() if near.any() else np.nan,
                    final_3d_signed_error=signed[near].mean() if near.any() else np.nan,
                    final_3d_pct_within_one=100 * (error[near] <= 1).mean() if near.any() else np.nan,
                    n_test_rows=len(test), n_near_rows=int(near.sum())))
                out = test[wearables.KEY + [TARGET]].copy()
                out = out.assign(model=label, leaf_size=leaf_size, seed=seed, prediction=pred,
                                 signed_error=signed, absolute_error=error)
                predictions.append(out)
            done = [r for r in scores if r["model"] == label and r["leaf_size"] == leaf_size]
            print(f"{label}, leaf {leaf_size}: train MAE {np.mean([r['train_MAE'] for r in done]):.3f}, "
                  f"held-out MAE {np.mean([r['test_MAE'] for r in done]):.3f}, "
                  f"final 3 days MAE {np.mean([r['final_3d_MAE'] for r in done]):.3f}", flush=True)
    return pd.DataFrame(scores), pd.concat(predictions, ignore_index=True)

leaf_scores, leaf_predictions = compare_leaf_sizes()
assert len(leaf_scores) == 2 * len(LEAF_SIZES) * 20
assert leaf_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
leaf_summary = leaf_scores.groupby(["model", "leaf_size"]).agg(
    train_MAE=("train_MAE", "mean"), inner_CV_MAE=("inner_CV_MAE", "mean"),
    heldout_MAE=("test_MAE", "mean"), heldout_split_SD=("test_MAE", "std"),
    heldout_participant_MAE=("test_participant_MAE", "mean"),
    final_3d_MAE=("final_3d_MAE", "mean"), final_3d_signed_error=("final_3d_signed_error", "mean"),
    final_3d_pct_within_one=("final_3d_pct_within_one", "mean"),
)
leaf_summary["test_minus_train_MAE"] = leaf_summary.heldout_MAE - leaf_summary.train_MAE
print("Mean across 20 splits; all MAEs in days. Training MAE is in-sample, not validation.")
with pd.option_context("display.max_columns", None):
    display(leaf_summary.round(3))
print("Selected depth counts (minimum leaf size is fixed in each experiment):")
display(pd.crosstab([leaf_scores.model, leaf_scores.leaf_size], leaf_scores.selected_depth))
print("Smaller minimum leaves permit smaller groups; actual leaves can be larger, especially with depths capped at 3 or 6.")
print("Exploratory comparison: selecting a winning leaf size from these test scores is not a final unbiased evaluation.")
# TODO(INTERVAL2_REVIEW): original data-quality caveats apply. No files exported.


In [ ]:
# Interval 2: RF squared-error vs absolute-error training criterion.
# Identical 20 participant splits, preprocessing, 100 trees and tuning grids.
# Both criteria are tuned using inner-validation MAE, never outer test outcomes.
# Wearables = all four latest values + original missing flags/ages, limited carry-forward.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend

if "fill_frames" not in globals():
    raise RuntimeError("Run the past-only imputation comparison setup first.")
loss_frame = fill_frames["limited_carry"].copy()
assert loss_frame.study_interval.eq(2024).all()
loss_grid = {"forest__max_depth": [3, 6], "forest__min_samples_leaf": [1, 3, 5, 10, 30]}

def compare_forest_losses(seeds=range(20)):
    scores, predictions = [], []
    for seed in seeds:
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(loss_frame)
        assert set(train.id).isdisjoint(test.id)
        folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(train, train[TARGET], groups=train.id))
        for model_name, features in [("history", []), ("history_wearables", fill_values + fill_context)]:
            columns = history_feature_columns + features
            for criterion in ["squared_error", "absolute_error"]:
                transforms = [("history", SimpleImputer(strategy="median", keep_empty_features=True), history_feature_columns)]
                if features:
                    transforms.append(("wearables", SimpleImputer(strategy="median", keep_empty_features=True), features))
                pipeline = Pipeline([
                    ("prepare", ColumnTransformer(transforms)),
                    ("forest", RandomForestRegressor(criterion=criterion, n_estimators=100,
                        max_features=1.0, random_state=SEED, n_jobs=1)),
                ])
                search = GridSearchCV(pipeline, loss_grid, scoring="neg_mean_absolute_error",
                                      cv=folds, n_jobs=4, error_score="raise")
                with parallel_backend("threading", n_jobs=4):
                    search.fit(train[columns], train[TARGET])
                pred = search.predict(test[columns])
                assert np.isfinite(pred).all()
                signed = pred - test[TARGET].to_numpy()
                error = np.abs(signed)
                near = test[TARGET].between(1, 3).to_numpy()
                scores.append(dict(model=model_name, criterion=criterion, seed=seed,
                    selected_depth=search.best_params_["forest__max_depth"],
                    selected_leaf=search.best_params_["forest__min_samples_leaf"],
                    train_MAE=np.abs(search.predict(train[columns]) - train[TARGET].to_numpy()).mean(),
                    inner_CV_MAE=-search.best_score_, test_MAE=error.mean(),
                    participant_MAE=pd.Series(error, index=test.id).groupby(level=0).mean().mean(),
                    final_3d_MAE=error[near].mean() if near.any() else np.nan,
                    final_3d_signed_error=signed[near].mean() if near.any() else np.nan,
                    final_3d_pct_within_one=100 * (error[near] <= 1).mean() if near.any() else np.nan,
                    n_test_rows=len(test)))
                out = test[wearables.KEY + [TARGET]].copy()
                predictions.append(out.assign(seed=seed, model=model_name, criterion=criterion,
                    prediction=pred, signed_error=signed, absolute_error=error))
        print(f"Loss comparison: split {seed + 1} finished", flush=True)
    return pd.DataFrame(scores), pd.concat(predictions, ignore_index=True)

loss_scores, loss_predictions = compare_forest_losses()
assert len(loss_scores) == 80
assert loss_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
loss_summary = loss_scores.groupby(["model", "criterion"]).agg(
    train_MAE=("train_MAE", "mean"), inner_CV_MAE=("inner_CV_MAE", "mean"),
    heldout_MAE=("test_MAE", "mean"), split_SD=("test_MAE", "std"),
    participant_MAE=("participant_MAE", "mean"), final_3d_MAE=("final_3d_MAE", "mean"),
    final_3d_signed_error=("final_3d_signed_error", "mean"),
    final_3d_pct_within_one=("final_3d_pct_within_one", "mean"),
)
print("Mean across 20 outer splits; positive signed error = late. All errors in days.")
with pd.option_context("display.max_columns", None):
    display(loss_summary.round(3))
loss_paired = loss_scores.pivot(index=["model", "seed"], columns="criterion", values="test_MAE")
loss_paired["absolute_minus_squared"] = loss_paired.absolute_error - loss_paired.squared_error
print("Paired criterion comparison: negative difference favors absolute-error training.")
display(loss_paired.groupby("model").agg(
    mean_MAE_change=("absolute_minus_squared", "mean"),
    absolute_error_better_splits=("absolute_minus_squared", lambda x: int(x.lt(0).sum())),
).round(4))
print("Selected parameter counts:")
display(loss_scores.groupby(["model", "criterion", "selected_depth", "selected_leaf"]).size().rename("splits"))
# Absolute-error trees use leaf medians, but forest predictions still average across trees.
# Leaf size/depth are tuned independently for each criterion; this compares tuned pipelines.
# Exploratory overlapping splits, not independent significance tests. TODO(INTERVAL2_REVIEW).
# No files exported; clear cell outputs before committing.


## Next Steps: Gradient Boosting

Might not be right for our dataset as it isn't neccessarily right for longitudinal data, but we'll use absolute-error loss instead our random forests squared error loss. 

In [ ]:
# Interval 2: gradient boosting with absolute-error loss, history vs history + wearables.
# Same 20 participant splits and limited-carry inputs as the preceding RF comparison.
# Inner GroupKFold selects depth, minimum leaf size and boosting stages using MAE.
# Fixed learning rate 0.05; no row-random early-stopping split for longitudinal records.
# TODO(INTERVAL2_REVIEW): unresolved dataset caveats still apply.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend

if "fill_frames" not in globals():
    raise RuntimeError("Run the past-only imputation comparison setup first.")
boost_frame = fill_frames["limited_carry"].copy()
assert boost_frame.study_interval.eq(2024).all()
boost_grid = {"boost__max_depth": [1, 3], "boost__min_samples_leaf": [5, 30], "boost__n_estimators": [100, 200]}

def compare_gradient_boosting(seeds=range(20)):
    scores, predictions = [], []
    for seed in seeds:
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(boost_frame)
        assert set(train.id).isdisjoint(test.id)
        folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(train, train[TARGET], groups=train.id))
        for model_name, features in [("history", []), ("history_wearables", fill_values + fill_context)]:
            columns = history_feature_columns + features
            for criterion in ["absolute_error"]:
                transforms = [("history", SimpleImputer(strategy="median", keep_empty_features=True), history_feature_columns)]
                if features:
                    transforms.append(("wearables", SimpleImputer(strategy="median", keep_empty_features=True), features))
                pipeline = Pipeline([
                    ("prepare", ColumnTransformer(transforms)),
                    ("boost", GradientBoostingRegressor(loss=criterion, learning_rate=0.05,
                        max_features=None, random_state=SEED, n_iter_no_change=None)),
                ])
                search = GridSearchCV(pipeline, boost_grid, scoring="neg_mean_absolute_error",
                                      cv=folds, n_jobs=4, error_score="raise")
                with parallel_backend("threading", n_jobs=4):
                    search.fit(train[columns], train[TARGET])
                pred = search.predict(test[columns])
                assert np.isfinite(pred).all()
                signed = pred - test[TARGET].to_numpy()
                error = np.abs(signed)
                near = test[TARGET].between(1, 3).to_numpy()
                scores.append(dict(model=model_name, criterion=criterion, seed=seed,
                    selected_depth=search.best_params_["boost__max_depth"],
                    selected_leaf=search.best_params_["boost__min_samples_leaf"],
                    selected_stages=search.best_params_["boost__n_estimators"],
                    train_MAE=np.abs(search.predict(train[columns]) - train[TARGET].to_numpy()).mean(),
                    inner_CV_MAE=-search.best_score_, test_MAE=error.mean(),
                    participant_MAE=pd.Series(error, index=test.id).groupby(level=0).mean().mean(),
                    final_3d_MAE=error[near].mean() if near.any() else np.nan,
                    final_3d_signed_error=signed[near].mean() if near.any() else np.nan,
                    final_3d_pct_within_one=100 * (error[near] <= 1).mean() if near.any() else np.nan,
                    n_test_rows=len(test)))
                out = test[wearables.KEY + [TARGET]].copy()
                predictions.append(out.assign(seed=seed, model=model_name, criterion=criterion,
                    prediction=pred, signed_error=signed, absolute_error=error))
        print(f"Absolute-error boosting: split {seed + 1} finished", flush=True)
    return pd.DataFrame(scores), pd.concat(predictions, ignore_index=True)

boost_scores, boost_predictions = compare_gradient_boosting()
assert len(boost_scores) == 40
assert boost_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
boost_summary = boost_scores.groupby(["model", "criterion"]).agg(
    train_MAE=("train_MAE", "mean"), inner_CV_MAE=("inner_CV_MAE", "mean"),
    heldout_MAE=("test_MAE", "mean"), split_SD=("test_MAE", "std"),
    participant_MAE=("participant_MAE", "mean"), final_3d_MAE=("final_3d_MAE", "mean"),
    final_3d_signed_error=("final_3d_signed_error", "mean"),
    final_3d_pct_within_one=("final_3d_pct_within_one", "mean"),
)
print("Mean across 20 outer splits; positive signed error = late. All errors in days.")
with pd.option_context("display.max_columns", None):
    display(boost_summary.round(3))

print("Selected boosting parameters:")
display(boost_scores.groupby(["model", "selected_depth", "selected_leaf", "selected_stages"]).size().rename("splits"))
boost_wearable_paired = boost_scores.pivot(index="seed", columns="model", values="test_MAE")
boost_wearable_change = boost_wearable_paired.history_wearables - boost_wearable_paired.history
print("Adding wearables: mean MAE change", round(boost_wearable_change.mean(), 4),
      "days; better in", int(boost_wearable_change.lt(0).sum()), "of 20 splits.")
if "loss_scores" in globals():
    boost_rf_comparison = boost_scores.merge(
        loss_scores.loc[loss_scores.criterion.eq("absolute_error")],
        on=["model", "seed"], suffixes=("_boost", "_rf"), validate="one_to_one",
    )
    assert len(boost_rf_comparison) == 40
    assert boost_rf_comparison.n_test_rows_boost.eq(boost_rf_comparison.n_test_rows_rf).all()
    boost_rf_comparison["boost_minus_RF"] = boost_rf_comparison.test_MAE_boost - boost_rf_comparison.test_MAE_rf
    print("Compared with preceding absolute-error RF (negative change favors boosting):")
    display(boost_rf_comparison.groupby("model").agg(
        boosting_MAE=("test_MAE_boost", "mean"), RF_MAE=("test_MAE_rf", "mean"),
        change_days=("boost_minus_RF", "mean"), boosting_better_splits=("boost_minus_RF", lambda x: int(x.lt(0).sum())),
    ).round(4))
else:
    print("Run the preceding RF loss comparison for the paired RF-versus-boosting table.")
# Exploratory comparisons on reused participants, not a final unbiased model-selection result.
# Predictions and parameters stay in memory; no files exported. Clear outputs before committing.


In [ ]:
# Interval 2: absolute-error gradient boosting with engineered breathing/temperature features.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend

# Interval 2 only. Preserve the previous feature definitions; no carry-forward.
# TODO(INTERVAL2_REVIEW): recheck dataset alignment/quality when clarification arrives.
engineered_boost_cycles = baseline_cycles.loc[baseline_cycles.study_interval.eq(2024)].copy()
engineered_boost_history = add_prior_cycle_history(
    data.make_daily_prediction_frame(engineered_boost_cycles), engineered_boost_cycles,
)
engineered_boost_daily = wearables.load_daily_signals()
engineered_boost_daily = engineered_boost_daily.loc[engineered_boost_daily.study_interval.eq(2024)].copy()
def build_engineered_boost_features(daily, frame):
    keys = pd.concat([daily[wearables.KEY], frame[wearables.KEY]]).drop_duplicates()
    dense = wearables.dense_days(keys).merge(
        daily[wearables.KEY + ["skin_temp", "breathing_rate"]],
        on=wearables.KEY, how="left", validate="one_to_one",
    )
    pieces = []
    for _, person in dense.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        result = person[wearables.KEY].copy()
        previous = person.breathing_rate.shift(1)
        window = previous.rolling(3, min_periods=3)  # D-3..D-1, complete calendar window.
        result["breathing_rate_3d_min"] = window.min()
        result["breathing_rate_3d_median"] = window.median()
        result["breathing_rate_3d_max"] = window.max()
        # Preserve the tested baseline: D-10..D-4, at least 4 valid days out of 7.
        result["temp_previous_vs_7d"] = (
            person.skin_temp.shift(1) - person.skin_temp.shift(4).rolling(7, min_periods=4).median()
        )
        pieces.append(result)
    return frame.merge(pd.concat(pieces, ignore_index=True), on=wearables.KEY,
                       how="left", validate="one_to_one")

engineered_boost_frame = build_engineered_boost_features(engineered_boost_daily, engineered_boost_history)
assert len(engineered_boost_frame) == len(engineered_boost_history)
assert engineered_boost_frame.study_interval.eq(2024).all()
engineered_boost_breathing = ["breathing_rate_3d_median", "breathing_rate_3d_min", "breathing_rate_3d_max"]
engineered_boost_temperature = ["temp_previous_vs_7d"]
engineered_boost_variants = {
    "history": [],
    "history_breathing": engineered_boost_breathing,
    "history_temperature": engineered_boost_temperature,
    "history_breathing_temperature": engineered_boost_breathing + engineered_boost_temperature,
}
# Require all 3 breathing days and at least 4 of 7 temperature-baseline days.
# Remaining missing features: training-fold median plus missingness indicators.
# Unlike the latest-value experiment, these summaries do not use limited carry-forward.
display(engineered_boost_frame[engineered_boost_breathing + engineered_boost_temperature]
        .isna().mean().mul(100).rename("missing_pct").round(1).to_frame())
engineered_boost_grid = {"boost__max_depth": [1, 3], "boost__min_samples_leaf": [5, 30], "boost__n_estimators": [100, 200]}

def compare_engineered_gradient_boosting(seeds=range(20)):
    scores, predictions = [], []
    for seed in seeds:
        split = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed)
        train, test = split.apply(engineered_boost_frame)
        assert set(train.id).isdisjoint(test.id)
        folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(train, train[TARGET], groups=train.id))
        for model_name, features in engineered_boost_variants.items():
            columns = history_feature_columns + features
            for criterion in ["absolute_error"]:
                transforms = [("history", SimpleImputer(strategy="median", keep_empty_features=True), history_feature_columns)]
                if features:
                    transforms.append(("wearables", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), features))
                pipeline = Pipeline([
                    ("prepare", ColumnTransformer(transforms)),
                    ("boost", GradientBoostingRegressor(loss=criterion, learning_rate=0.05,
                        max_features=None, random_state=SEED, n_iter_no_change=None)),
                ])
                search = GridSearchCV(pipeline, engineered_boost_grid, scoring="neg_mean_absolute_error",
                                      cv=folds, n_jobs=4, error_score="raise")
                with parallel_backend("threading", n_jobs=4):
                    search.fit(train[columns], train[TARGET])
                pred = search.predict(test[columns])
                assert np.isfinite(pred).all()
                signed = pred - test[TARGET].to_numpy()
                error = np.abs(signed)
                near = test[TARGET].between(1, 3).to_numpy()
                scores.append(dict(model=model_name, criterion=criterion, seed=seed,
                    selected_depth=search.best_params_["boost__max_depth"],
                    selected_leaf=search.best_params_["boost__min_samples_leaf"],
                    selected_stages=search.best_params_["boost__n_estimators"],
                    train_MAE=np.abs(search.predict(train[columns]) - train[TARGET].to_numpy()).mean(),
                    inner_CV_MAE=-search.best_score_, test_MAE=error.mean(),
                    participant_MAE=pd.Series(error, index=test.id).groupby(level=0).mean().mean(),
                    final_3d_MAE=error[near].mean() if near.any() else np.nan,
                    final_3d_signed_error=signed[near].mean() if near.any() else np.nan,
                    final_3d_pct_within_one=100 * (error[near] <= 1).mean() if near.any() else np.nan,
                    n_test_rows=len(test)))
                out = test[wearables.KEY + [TARGET]].copy()
                predictions.append(out.assign(seed=seed, model=model_name, criterion=criterion,
                    prediction=pred, signed_error=signed, absolute_error=error))
        print(f"Absolute-error boosting: split {seed + 1} finished", flush=True)
    return pd.DataFrame(scores), pd.concat(predictions, ignore_index=True)

engineered_boost_scores, engineered_boost_predictions = compare_engineered_gradient_boosting()
assert len(engineered_boost_scores) == 80
assert engineered_boost_scores.groupby("seed").n_test_rows.nunique().eq(1).all()
engineered_boost_summary = engineered_boost_scores.groupby(["model", "criterion"]).agg(
    train_MAE=("train_MAE", "mean"), inner_CV_MAE=("inner_CV_MAE", "mean"),
    heldout_MAE=("test_MAE", "mean"), split_SD=("test_MAE", "std"),
    participant_MAE=("participant_MAE", "mean"), final_3d_MAE=("final_3d_MAE", "mean"),
    final_3d_signed_error=("final_3d_signed_error", "mean"),
    final_3d_pct_within_one=("final_3d_pct_within_one", "mean"),
)
print("Mean across 20 outer splits; positive signed error = late. All errors in days.")
with pd.option_context("display.max_columns", None):
    display(engineered_boost_summary.round(3))

print("Selected boosting parameters:")
display(engineered_boost_scores.groupby(["model", "selected_depth", "selected_leaf", "selected_stages"]).size().rename("splits"))
engineered_boost_paired = engineered_boost_scores.pivot(index="seed", columns="model", values="test_MAE")
engineered_boost_changes = engineered_boost_paired.subtract(engineered_boost_paired["history"], axis=0)
engineered_boost_comparison = pd.DataFrame({
    "heldout_MAE": engineered_boost_paired.mean(),
    "change_vs_history": engineered_boost_changes.mean(),
    "better_than_history_splits": engineered_boost_changes.lt(0).sum(),
})
print("Same held-out rows in every variant; negative change favors adding features:")
display(engineered_boost_comparison.round(4))
# Exploratory feature comparisons on reused splits, not an unbiased final evaluation.
# Model settings and imputation are fitted inside participant-grouped inner validation.
# All predictions remain in memory; no exports. Clear outputs before committing.


Last chance before trying a state space model, switch temp baseline definition to D-8 through D-2 compared to D-1.

In [ ]:
# Temperature endpoint comparison: previous day versus immediately preceding week.
# Preserve earlier experiments. New feature = T[D-1] - median(T[D-8:D-2]).
# Old feature = T[D-1] - median(T[D-10:D-4]); both require 4/7 valid days.
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV
from joblib import parallel_backend
from IPython.display import display

def build_temperature_endpoint_features(daily, frame):
    keys = pd.concat([daily[wearables.KEY], frame[wearables.KEY]]).drop_duplicates()
    dense = wearables.dense_days(keys).merge(
        daily[wearables.KEY + ["skin_temp"]], on=wearables.KEY,
        how="left", validate="one_to_one",
    )
    pieces = []
    for _, person in dense.groupby(wearables.PARTICIPANT, sort=False):
        person = person.sort_values("day_in_study")
        out = person[wearables.KEY].copy()
        temperature = person.skin_temp
        out["temp_7d_old_gap"] = temperature.shift(1) - temperature.shift(4).rolling(7, min_periods=4).median()
        out["temp_7d_preceding_week"] = temperature.shift(1) - temperature.shift(2).rolling(7, min_periods=4).median()
        pieces.append(out)
    return frame.merge(pd.concat(pieces, ignore_index=True), on=wearables.KEY,
                       how="left", validate="one_to_one")

# No carry-forward, no current-day readings; missing values remain missing until
# training-fold imputation. Group windows by participant AND study interval.
endpoint_daily = wearables.load_daily_signals()
endpoint_history = add_prior_cycle_history(data.make_daily_prediction_frame(baseline_cycles), baseline_cycles)
endpoint_frame = build_temperature_endpoint_features(endpoint_daily, endpoint_history)
endpoint_variants = {
    "history": [],
    "old_gap": ["temp_7d_old_gap"],
    "preceding_week": ["temp_7d_preceding_week"],
}
endpoint_experiments = {
    "RF_interval1": ((2022,), "forest"),
    "RF_both_intervals": ((2022, 2024), "forest"),
    "GB_absolute_interval2": ((2024,), "boost"),
}

def compare_temperature_endpoints(seeds=range(20)):
    scores, predictions = [], []
    for experiment, (intervals, family) in endpoint_experiments.items():
        frame = endpoint_frame.loc[endpoint_frame.study_interval.isin(intervals)].copy()
        for seed in seeds:
            train, test = data.participant_split(baseline_cycles, test_frac=0.3, seed=seed).apply(frame)
            assert set(train.id).isdisjoint(test.id)
            folds = list(GroupKFold(n_splits=min(5, train.id.nunique())).split(train, train[TARGET], groups=train.id))
            for variant, features in endpoint_variants.items():
                columns = history_feature_columns + features
                transforms = [("history", SimpleImputer(strategy="median", keep_empty_features=True), history_feature_columns)]
                if features:
                    transforms.append(("temperature", SimpleImputer(strategy="median", add_indicator=True, keep_empty_features=True), features))
                if family == "forest":
                    estimator = RandomForestRegressor(n_estimators=100, max_features=1.0, random_state=SEED, n_jobs=1)
                    grid = {"model__max_depth": [3, 6], "model__min_samples_leaf": [10, 30]}
                else:
                    estimator = GradientBoostingRegressor(loss="absolute_error", learning_rate=0.05, max_features=None, random_state=SEED, n_iter_no_change=None)
                    grid = {"model__max_depth": [1, 3], "model__min_samples_leaf": [5, 30], "model__n_estimators": [100, 200]}
                search = GridSearchCV(Pipeline([("prepare", ColumnTransformer(transforms)), ("model", estimator)]),
                                      grid, scoring="neg_mean_absolute_error", cv=folds, n_jobs=4, error_score="raise")
                with parallel_backend("threading", n_jobs=4):
                    search.fit(train[columns], train[TARGET])
                pred = search.predict(test[columns])
                assert np.isfinite(pred).all()
                signed = pred - test[TARGET].to_numpy()
                error = np.abs(signed)
                near = test[TARGET].between(1, 3).to_numpy()
                scores.append(dict(experiment=experiment, variant=variant, seed=seed,
                    MAE=error.mean(), participant_MAE=pd.Series(error, index=test.id).groupby(level=0).mean().mean(),
                    final_3d_MAE=error[near].mean(), final_3d_signed_error=signed[near].mean(),
                    inner_CV_MAE=-search.best_score_, selected_parameters=search.best_params_,
                    n_test_rows=len(test), missing_pct=100 * test[features].isna().any(axis=1).mean() if features else np.nan))
                predictions.append(test[wearables.KEY + [TARGET]].assign(experiment=experiment,
                    variant=variant, seed=seed, prediction=pred, signed_error=signed, absolute_error=error))
            print(f"{experiment}: split {seed + 1} finished", flush=True)
    return pd.DataFrame(scores), pd.concat(predictions, ignore_index=True)

endpoint_scores, endpoint_predictions = compare_temperature_endpoints()
assert len(endpoint_scores) == 180
assert endpoint_scores.groupby(["experiment", "seed"]).n_test_rows.nunique().eq(1).all()
endpoint_summary = endpoint_scores.groupby(["experiment", "variant"]).agg(
    mean_MAE=("MAE", "mean"), split_SD=("MAE", "std"),
    participant_MAE=("participant_MAE", "mean"), final_3d_MAE=("final_3d_MAE", "mean"),
    final_3d_signed_error=("final_3d_signed_error", "mean"), missing_pct=("missing_pct", "mean"),
)
endpoint_paired = endpoint_scores.pivot(index=["experiment", "seed"], columns="variant", values="MAE")
endpoint_changes = pd.DataFrame({
    "new_minus_old": endpoint_paired.preceding_week - endpoint_paired.old_gap,
    "new_minus_history": endpoint_paired.preceding_week - endpoint_paired.history,
})
print("MAE in days; negative paired change is better. Split SD is not a confidence interval.")
with pd.option_context("display.max_columns", None):
    display(endpoint_summary.round(4))
    display(endpoint_changes.groupby("experiment").agg(["mean", lambda s: int(s.lt(0).sum())]).rename(columns={"<lambda_0>": "improved_splits"}).round(4))
# Exploratory reused outer splits; parameters and imputation fit inside grouped inner folds.
# TODO(INTERVAL2_REVIEW): phase-derived labels and data processing remain provisional.
# No files exported. Keep participant-level predictions and notebook outputs private.
